# Integrated Alignment Audit

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 30.


In [ ]:
# ============================================================
# INTEGRATED FINAL — STEP 2A
# ALIGNMENT + FILE DISCOVERY AUDIT
# ============================================================
#
# Goal:
# 1) Inspect main Interaction train/test files
# 2) Find saved Interaction predictions/probabilities
# 3) Find Side-Effect train/test row metadata
# 4) Check pair keys and row counts
# 5) Prepare safe integration without retraining
#
# IMPORTANT:
# - NO TRAINING
# - NO ROW REORDERING YET
# - NO ASSUMED ALIGNMENT
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import os
import json
import warnings

warnings.filterwarnings("ignore")


# ============================================================
# 1) ROOTS
# ============================================================

INTERACTION_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-v2-menna-projectt"
)

SIDE_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)


print("=" * 120)
print("ROOT CHECK")
print("=" * 120)

print("INTERACTION_ROOT:")
print(INTERACTION_ROOT)
print("Exists:", INTERACTION_ROOT.exists())

print("\nSIDE_ROOT:")
print(SIDE_ROOT)
print("Exists:", SIDE_ROOT.exists())

assert INTERACTION_ROOT.exists()
assert SIDE_ROOT.exists()


# ============================================================
# 2) KNOWN MAIN INTERACTION SPLITS
# ============================================================

INT_TRAIN = (
    INTERACTION_ROOT /
    "clean_train_split.csv"
)

INT_CAL = (
    INTERACTION_ROOT /
    "clean_calibration_split.csv"
)

INT_TEST = (
    INTERACTION_ROOT /
    "clean_test_split.csv"
)


print("\n" + "=" * 120)
print("MAIN INTERACTION SPLITS")
print("=" * 120)


for name, path in [
    ("TRAIN", INT_TRAIN),
    ("CAL", INT_CAL),
    ("TEST", INT_TEST)
]:

    print(
        f"{name:6s}: "
        f"{path}"
    )

    print(
        "Exists:",
        path.exists()
    )


assert INT_TRAIN.exists()
assert INT_TEST.exists()


# ============================================================
# 3) LOAD MAIN INTERACTION SPLITS
# ============================================================

interaction_train = pd.read_csv(
    INT_TRAIN
)

interaction_test = pd.read_csv(
    INT_TEST
)


if INT_CAL.exists():

    interaction_cal = pd.read_csv(
        INT_CAL
    )

else:

    interaction_cal = None


print("\n" + "=" * 120)
print("INTERACTION SPLIT SUMMARY")
print("=" * 120)


print(
    "Train shape:",
    interaction_train.shape
)

print(
    "Test shape:",
    interaction_test.shape
)

if interaction_cal is not None:

    print(
        "Calibration shape:",
        interaction_cal.shape
    )


print("\nInteraction train columns:")
print(
    interaction_train.columns.tolist()
)

print("\nInteraction test columns:")
print(
    interaction_test.columns.tolist()
)


# ============================================================
# 4) SHOW IMPORTANT INTERACTION COLUMNS
# ============================================================

IMPORTANT_NAMES = [

    "Drug1_ID",
    "Drug2_ID",
    "Drug1",
    "Drug2",
    "Interaction",
    "Label",
    "label",
    "interaction_id",
    "Interaction_ID",
    "target",
    "y"
]


def existing_columns(
    df,
    names
):

    return [
        c
        for c in names
        if c in df.columns
    ]


important_int_cols = existing_columns(
    interaction_test,
    IMPORTANT_NAMES
)


print("\nImportant Interaction test columns:")
print(
    important_int_cols
)


if important_int_cols:

    print("\nInteraction test preview:")

    print(
        interaction_test[
            important_int_cols
        ]
        .head(10)
        .to_string(
            index=False
        )
    )


# ============================================================
# 5) SIDE-EFFECT TARGET FILES
# ============================================================

SIDE_Y_TRAIN = (
    SIDE_ROOT /
    "Y_train (1).npy"
)

SIDE_Y_CAL = (
    SIDE_ROOT /
    "Y_calibration.npy"
)

SIDE_Y_TEST = (
    SIDE_ROOT /
    "Y_test.npy"
)


print("\n" + "=" * 120)
print("SIDE-EFFECT TARGETS")
print("=" * 120)


for name, path in [

    ("Y_train", SIDE_Y_TRAIN),
    ("Y_cal", SIDE_Y_CAL),
    ("Y_test", SIDE_Y_TEST)

]:

    print(
        name,
        "->",
        path,
        "| exists =",
        path.exists()
    )


assert SIDE_Y_TRAIN.exists()
assert SIDE_Y_CAL.exists()
assert SIDE_Y_TEST.exists()


Y_train = np.load(
    SIDE_Y_TRAIN,
    mmap_mode="r"
)

Y_cal = np.load(
    SIDE_Y_CAL,
    mmap_mode="r"
)

Y_test = np.load(
    SIDE_Y_TEST,
    mmap_mode="r"
)


print("\nShapes:")

print(
    "Y_train:",
    Y_train.shape
)

print(
    "Y_cal:",
    Y_cal.shape
)

print(
    "Y_test:",
    Y_test.shape
)


assert Y_train.shape == (
    134249,
    263
)

assert Y_cal.shape == (
    28768,
    263
)

assert Y_test.shape == (
    28768,
    263
)


# ============================================================
# 6) SIDE-EFFECT PRIMARY PROBABILITIES
# ============================================================

BASELINE_ROOT = (
    SIDE_ROOT /
    "PRIMARY_UNIQUE263_BASELINE_PROBS"
)


print("\n" + "=" * 120)
print("SIDE-EFFECT PRIMARY ENSEMBLE FILES")
print("=" * 120)


for seed in [
    42,
    43,
    44
]:

    d = (
        BASELINE_ROOT /
        f"seed_{seed}"
    )

    cal_p = (
        d /
        "calibration_probabilities.npy"
    )

    test_p = (
        d /
        "test_probabilities.npy"
    )

    print(
        f"Seed {seed}:"
    )

    print(
        "  Cal :",
        cal_p.exists(),
        cal_p
    )

    print(
        "  Test:",
        test_p.exists(),
        test_p
    )


# ============================================================
# 7) SEARCH SIDE-EFFECT ROW / SPLIT METADATA
# ============================================================

print("\n" + "=" * 120)
print("SIDE-EFFECT ROW / SPLIT METADATA CANDIDATES")
print("=" * 120)


side_candidate_files = []


KEYWORDS = [

    "train",
    "test",
    "cal",
    "split",
    "row",
    "index",
    "indice",
    "pair"
]


for p in SIDE_ROOT.rglob("*"):

    if not p.is_file():
        continue

    name = p.name.lower()

    if not (
        name.endswith(".csv")
        or
        name.endswith(".npy")
        or
        name.endswith(".npz")
        or
        name.endswith(".json")
    ):
        continue

    if any(
        k in name
        for k in KEYWORDS
    ):

        # Skip known probability files here
        if "probabilit" in name:
            continue

        side_candidate_files.append(
            p
        )


for p in sorted(
    side_candidate_files
):

    print(
        " •",
        p.relative_to(
            SIDE_ROOT
        )
    )


# ============================================================
# 8) INSPECT SIDE-EFFECT CSV CANDIDATES
# ============================================================

print("\n" + "=" * 120)
print("SIDE-EFFECT CSV CANDIDATE DETAILS")
print("=" * 120)


side_csv_summary = []


for p in side_candidate_files:

    if p.suffix.lower() != ".csv":
        continue

    try:

        df = pd.read_csv(
            p
        )

        cols = df.columns.tolist()

        pair_cols = [
            c
            for c in [
                "Drug1_ID",
                "Drug2_ID",
                "Drug1",
                "Drug2",
                "Interaction",
                "Label",
                "Interaction_ID"
            ]
            if c in cols
        ]


        side_csv_summary.append({

            "file":
                str(
                    p.relative_to(
                        SIDE_ROOT
                    )
                ),

            "rows":
                len(
                    df
                ),

            "cols":
                len(
                    cols
                ),

            "pair_columns":
                ", ".join(
                    pair_cols
                )
        })

    except Exception as e:

        pass


side_csv_summary = pd.DataFrame(
    side_csv_summary
)


if len(
    side_csv_summary
):

    side_csv_summary = (
        side_csv_summary
        .sort_values(
            [
                "rows",
                "file"
            ]
        )
    )


    print(
        side_csv_summary.to_string(
            index=False
        )
    )

else:

    print(
        "No matching side-effect CSV metadata files found."
    )


# ============================================================
# 9) SEARCH MAIN INTERACTION PREDICTION FILES
# ============================================================

print("\n" + "=" * 120)
print("SEARCHING MAIN INTERACTION PREDICTIONS")
print("=" * 120)


# Exclude alternative experiments:
# cold-start, scaffold, AutoDDI, MHAFR, etc.
EXCLUDE_TOKENS = [

    "cold",
    "scaffold",
    "autoddi",
    "mhafr",
    "mkg",
    "3d_candidates",
    "coordinates"
]


PREDICTION_KEYWORDS = [

    "pred",
    "prob",
    "logit",
    "output",
    "result"
]


interaction_prediction_candidates = []


for p in INTERACTION_ROOT.rglob("*"):

    if not p.is_file():
        continue

    s = str(
        p.relative_to(
            INTERACTION_ROOT
        )
    ).lower()

    name = p.name.lower()


    if any(
        token in s
        for token in EXCLUDE_TOKENS
    ):

        continue


    if not (
        name.endswith(".csv")
        or
        name.endswith(".npy")
        or
        name.endswith(".npz")
        or
        name.endswith(".json")
    ):

        continue


    if any(
        k in name
        for k in PREDICTION_KEYWORDS
    ):

        interaction_prediction_candidates.append(
            p
        )


if interaction_prediction_candidates:

    for p in sorted(
        interaction_prediction_candidates
    ):

        print(
            " •",
            p.relative_to(
                INTERACTION_ROOT
            )
        )

else:

    print(
        " No top-level/main interaction prediction file found."
    )


# ============================================================
# 10) INSPECT INTERACTION PREDICTION CSVs
# ============================================================

print("\n" + "=" * 120)
print("INTERACTION PREDICTION CSV DETAILS")
print("=" * 120)


interaction_pred_summary = []


for p in interaction_prediction_candidates:

    if p.suffix.lower() != ".csv":
        continue

    try:

        df = pd.read_csv(
            p
        )

        interaction_pred_summary.append({

            "file":
                str(
                    p.relative_to(
                        INTERACTION_ROOT
                    )
                ),

            "rows":
                len(
                    df
                ),

            "columns":
                " | ".join(
                    df.columns.tolist()
                )
        })

    except Exception:
        pass


if interaction_pred_summary:

    interaction_pred_summary = pd.DataFrame(
        interaction_pred_summary
    )

    print(
        interaction_pred_summary.to_string(
            index=False
        )
    )

else:

    print(
        "No candidate interaction prediction CSVs."
    )


# ============================================================
# 11) INSPECT INTERACTION NPY / NPZ CANDIDATES
# ============================================================

print("\n" + "=" * 120)
print("INTERACTION PREDICTION ARRAY DETAILS")
print("=" * 120)


for p in interaction_prediction_candidates:

    if p.suffix.lower() not in [
        ".npy",
        ".npz"
    ]:

        continue


    try:

        if p.suffix.lower() == ".npy":

            arr = np.load(
                p,
                mmap_mode="r"
            )

            print(
                p.relative_to(
                    INTERACTION_ROOT
                ),
                "->",
                arr.shape,
                arr.dtype
            )


        else:

            z = np.load(
                p
            )

            print(
                "\n",
                p.relative_to(
                    INTERACTION_ROOT
                )
            )

            print(
                "keys:",
                z.files
            )

            for key in z.files[:10]:

                arr = z[
                    key
                ]

                print(
                    "   ",
                    key,
                    "->",
                    arr.shape,
                    arr.dtype
                )


    except Exception as e:

        print(
            "Could not inspect:",
            p,
            e
        )


# ============================================================
# 12) CHECK WHETHER MAIN INTERACTION TEST SIZE
#     MATCHES SIDE-EFFECT TEST SIZE
# ============================================================

print("\n" + "=" * 120)
print("TEST SIZE COMPARISON")
print("=" * 120)


n_int_test = len(
    interaction_test
)

n_side_test = Y_test.shape[
    0
]


print(
    "Interaction test rows:",
    n_int_test
)

print(
    "Side-effect test rows:",
    n_side_test
)

print(
    "Same row count:",
    n_int_test
    ==
    n_side_test
)


# ============================================================
# 13) PAIR KEY QA FOR INTERACTION SPLITS
# ============================================================

print("\n" + "=" * 120)
print("INTERACTION PAIR KEY QA")
print("=" * 120)


PAIR_ID_COLS = [
    "Drug1_ID",
    "Drug2_ID"
]


PAIR_NAME_COLS = [
    "Drug1",
    "Drug2"
]


if all(
    c in interaction_test.columns
    for c in PAIR_ID_COLS
):

    print(
        "Using Drug1_ID + Drug2_ID as ordered pair key."
    )


    print(
        "Unique ordered train pairs:",
        interaction_train[
            PAIR_ID_COLS
        ]
        .drop_duplicates()
        .shape[
            0
        ]
    )


    print(
        "Unique ordered test pairs:",
        interaction_test[
            PAIR_ID_COLS
        ]
        .drop_duplicates()
        .shape[
            0
        ]
    )


    train_pairs = set(
        map(
            tuple,
            interaction_train[
                PAIR_ID_COLS
            ].astype(str).values
        )
    )


    test_pairs = list(
        map(
            tuple,
            interaction_test[
                PAIR_ID_COLS
            ].astype(str).values
        )
    )


    seen_test_rows = np.array(
        [
            p in train_pairs
            for p in test_pairs
        ]
    )


    print(
        "Test rows whose ordered pair appears in train:",
        int(
            seen_test_rows.sum()
        ),
        "/",
        len(
            seen_test_rows
        )
    )


    print(
        "Percentage:",
        f"{100*seen_test_rows.mean():.4f}%"
    )


elif all(
    c in interaction_test.columns
    for c in PAIR_NAME_COLS
):

    print(
        "Drug IDs unavailable; using ordered Drug1 + Drug2 names."
    )


else:

    print(
        " No obvious ordered-pair columns found."
    )


# ============================================================
# 14) LOOK FOR CURRENT FINAL SIDE-EFFECT DATASET
# ============================================================

CURRENT_SIDE_DATA = (
    SIDE_ROOT /
    "cleaned_ddi_sideeffects_84_263_UNIQUE.csv"
)


print("\n" + "=" * 120)
print("CURRENT FINAL SIDE-EFFECT DATASET")
print("=" * 120)


print(
    CURRENT_SIDE_DATA
)

print(
    "Exists:",
    CURRENT_SIDE_DATA.exists()
)


if CURRENT_SIDE_DATA.exists():

    side_full = pd.read_csv(
        CURRENT_SIDE_DATA
    )


    print(
        "Shape:",
        side_full.shape
    )

    print(
        "Columns:",
        side_full.columns.tolist()
    )


    show_cols = [
        c
        for c in [
            "Drug1_ID",
            "Drug1",
            "Drug2_ID",
            "Drug2",
            "Label",
            "Interaction",
            "sideeffect_str"
        ]
        if c in side_full.columns
    ]


    print(
        "\nPreview:"
    )

    print(
        side_full[
            show_cols
        ]
        .head(
            5
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 15) SEARCH FOR SAVED SPLIT INDICES
# ============================================================

print("\n" + "=" * 120)
print("SIDE-EFFECT SPLIT INDEX CANDIDATES")
print("=" * 120)


index_candidates = []


for p in SIDE_ROOT.rglob("*"):

    if not p.is_file():
        continue

    name = p.name.lower()

    if (
        "index" in name
        or
        "indices" in name
        or
        "row" in name
        or
        "split" in name
    ):

        if p.suffix.lower() in [
            ".npy",
            ".npz",
            ".csv",
            ".json"
        ]:

            index_candidates.append(
                p
            )


if index_candidates:

    for p in sorted(
        index_candidates
    ):

        print(
            " •",
            p.relative_to(
                SIDE_ROOT
            )
        )

else:

    print(
        "No explicit split-index files found."
    )


# ============================================================
# 16) FINAL AUDIT SUMMARY
# ============================================================

print("\n" + "=" * 120)
print("INTEGRATED AUDIT SUMMARY")
print("=" * 120)


print(
    "Interaction train rows :",
    len(
        interaction_train
    )
)


print(
    "Interaction test rows  :",
    len(
        interaction_test
    )
)


print(
    "SideEffect train rows  :",
    Y_train.shape[
        0
    ]
)


print(
    "SideEffect test rows   :",
    Y_test.shape[
        0
    ]
)


print(
    "\nMain interaction prediction candidates:",
    len(
        interaction_prediction_candidates
    )
)


print(
    "Side-effect metadata candidate files:",
    len(
        side_candidate_files
    )
)


print("\n NO TRAINING PERFORMED")
print(" No integrated metric calculated before row alignment")
print(" Ordered Drug1→Drug2 direction preserved")

Source cell: 31.


In [ ]:
# ============================================================
# INTEGRATED FINAL — STEP 2B
# ROW-ALIGNMENT PROOF + INTERACTION CHECKPOINT AUDIT
# ============================================================
#
# NO TRAINING
#
# This cell:
# 1) Reconstructs ordered Morgan fingerprints from
#    Interaction clean_test_split.csv
# 2) Compares them EXACTLY with Side-Effect X_test.npz
# 3) Repeats alignment check on calibration split
# 4) Cross-matches rows against final Side-Effect dataset
# 5) Audits BEST_checkpoint.pt
# 6) Audits feature cache / metadata needed for inference
#
# ============================================================

from pathlib import Path
import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

warnings.filterwarnings("ignore")


# ============================================================
# 1) PATHS
# ============================================================

INTERACTION_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-v2-menna-projectt"
)

SIDE_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

INT_TRAIN_PATH = (
    INTERACTION_ROOT /
    "clean_train_split.csv"
)

INT_CAL_PATH = (
    INTERACTION_ROOT /
    "clean_calibration_split.csv"
)

INT_TEST_PATH = (
    INTERACTION_ROOT /
    "clean_test_split.csv"
)

CHECKPOINT_PATH = (
    INTERACTION_ROOT /
    "BEST_checkpoint.pt"
)

FEATURE_CACHE_PATH = (
    INTERACTION_ROOT /
    "exact_v2_feature_cache_2249.npz"
)

FEATURE_META_PATH = (
    INTERACTION_ROOT /
    "exact_v2_feature_meta.json"
)

INTERACTION_LABEL_MAP = (
    INTERACTION_ROOT /
    "label_mapping.csv"
)

SIDE_FULL_PATH = (
    SIDE_ROOT /
    "cleaned_ddi_sideeffects_84_263_UNIQUE.csv"
)

SIDE_X_CAL_PATH = (
    SIDE_ROOT /
    "X_calibration.npz"
)

SIDE_X_TEST_PATH = (
    SIDE_ROOT /
    "X_test.npz"
)

SIDE_Y_CAL_PATH = (
    SIDE_ROOT /
    "Y_calibration.npy"
)

SIDE_Y_TEST_PATH = (
    SIDE_ROOT /
    "Y_test.npy"
)


# ============================================================
# 2) PATH QA
# ============================================================

print("=" * 120)
print("PATH QA")
print("=" * 120)

paths = {
    "Interaction train": INT_TRAIN_PATH,
    "Interaction cal": INT_CAL_PATH,
    "Interaction test": INT_TEST_PATH,
    "BEST checkpoint": CHECKPOINT_PATH,
    "Feature cache": FEATURE_CACHE_PATH,
    "Feature meta": FEATURE_META_PATH,
    "Interaction label map": INTERACTION_LABEL_MAP,
    "Final SideEffect dataset": SIDE_FULL_PATH,
    "Side X_cal": SIDE_X_CAL_PATH,
    "Side X_test": SIDE_X_TEST_PATH,
    "Side Y_cal": SIDE_Y_CAL_PATH,
    "Side Y_test": SIDE_Y_TEST_PATH,
}

for name, p in paths.items():
    print(
        f"{name:26s}: "
        f"{p.exists()}  "
        f"{p}"
    )


required = [
    INT_TRAIN_PATH,
    INT_CAL_PATH,
    INT_TEST_PATH,
    CHECKPOINT_PATH,
    SIDE_FULL_PATH,
    SIDE_X_CAL_PATH,
    SIDE_X_TEST_PATH,
    SIDE_Y_CAL_PATH,
    SIDE_Y_TEST_PATH,
]

for p in required:
    assert p.exists(), p


# ============================================================
# 3) LOAD SPLITS
# ============================================================

train_df = pd.read_csv(
    INT_TRAIN_PATH
)

cal_df = pd.read_csv(
    INT_CAL_PATH
)

test_df = pd.read_csv(
    INT_TEST_PATH
)

side_full = pd.read_csv(
    SIDE_FULL_PATH
)

Y_cal = np.load(
    SIDE_Y_CAL_PATH,
    mmap_mode="r"
)

Y_test = np.load(
    SIDE_Y_TEST_PATH,
    mmap_mode="r"
)

X_cal = sp.load_npz(
    SIDE_X_CAL_PATH
).tocsr()

X_test = sp.load_npz(
    SIDE_X_TEST_PATH
).tocsr()


print("\n" + "=" * 120)
print("SPLIT SHAPES")
print("=" * 120)

print("Interaction train:", train_df.shape)
print("Interaction cal  :", cal_df.shape)
print("Interaction test :", test_df.shape)

print("Side full        :", side_full.shape)

print("X_cal            :", X_cal.shape)
print("X_test           :", X_test.shape)

print("Y_cal            :", Y_cal.shape)
print("Y_test           :", Y_test.shape)


assert len(cal_df) == X_cal.shape[0] == Y_cal.shape[0]
assert len(test_df) == X_test.shape[0] == Y_test.shape[0]

assert X_cal.shape[1] == 4096
assert X_test.shape[1] == 4096

assert Y_cal.shape[1] == 263
assert Y_test.shape[1] == 263


# ============================================================
# 4) BASIC SPLIT LABEL QA
# ============================================================

print("\n" + "=" * 120)
print("INTERACTION LABEL QA")
print("=" * 120)

for name, df in [
    ("TRAIN", train_df),
    ("CAL", cal_df),
    ("TEST", test_df),
]:

    print(
        name,
        "rows =",
        len(df),
        "| classes =",
        df["label_id"].nunique(),
        "| exact labels =",
        df["Label"].nunique()
    )


print(
    "\nUnique total split rows:",
    len(train_df) + len(cal_df) + len(test_df)
)

print(
    "Final side-effect rows:",
    len(side_full)
)


# ============================================================
# 5) ORDERED-PAIR OVERLAP AUDIT
# ============================================================

# Drug1 and Drug2 here are SMILES.
# ORDER IS PRESERVED.
def pair_series(df):
    return list(
        zip(
            df["Drug1"].astype(str),
            df["Drug2"].astype(str)
        )
    )


train_pairs = set(
    pair_series(train_df)
)

cal_pairs = set(
    pair_series(cal_df)
)

test_pairs = set(
    pair_series(test_df)
)


train_test_overlap = (
    train_pairs
    &
    test_pairs
)

train_cal_overlap = (
    train_pairs
    &
    cal_pairs
)

cal_test_overlap = (
    cal_pairs
    &
    test_pairs
)


print("\n" + "=" * 120)
print("ORDERED PAIR OVERLAP")
print("=" * 120)

print(
    "Unique train pairs:",
    len(train_pairs)
)

print(
    "Unique cal pairs  :",
    len(cal_pairs)
)

print(
    "Unique test pairs :",
    len(test_pairs)
)

print(
    "\nTrain ∩ Test:",
    len(train_test_overlap)
)

print(
    "Train ∩ Cal :",
    len(train_cal_overlap)
)

print(
    "Cal ∩ Test  :",
    len(cal_test_overlap)
)


# ============================================================
# 6) BUILD MORGAN GENERATOR
# ============================================================

# Must match the final Side-Effect representation:
# radius = 2
# 2048 bits per drug
# no chirality
# Drug1 || Drug2 = 4096

morgan_gen = (
    rdFingerprintGenerator.GetMorganGenerator(
        radius=2,
        fpSize=2048,
        includeChirality=False
    )
)


fp_cache = {}


def smiles_to_fp(smiles):

    smiles = str(
        smiles
    )

    if smiles in fp_cache:
        return fp_cache[smiles]

    mol = Chem.MolFromSmiles(
        smiles
    )

    if mol is None:
        raise ValueError(
            f"Invalid SMILES: {smiles}"
        )

    fp = morgan_gen.GetFingerprintAsNumPy(
        mol
    ).astype(
        np.uint8
    )

    assert fp.shape == (2048,)

    fp_cache[smiles] = fp

    return fp


def reconstruct_pair_matrix(df):

    rows = []

    for i, row in enumerate(
        df.itertuples(index=False)
    ):

        fp1 = smiles_to_fp(
            row.Drug1
        )

        fp2 = smiles_to_fp(
            row.Drug2
        )

        pair_fp = np.concatenate(
            [
                fp1,
                fp2
            ]
        )

        rows.append(
            pair_fp
        )

        if (
            (i + 1) % 5000
            ==
            0
        ):
            print(
                f"  reconstructed "
                f"{i+1:,}/{len(df):,}"
            )

    X = np.asarray(
        rows,
        dtype=np.uint8
    )

    return sp.csr_matrix(
        X
    )


# ============================================================
# 7) EXACT SPARSE MATRIX COMPARISON
# ============================================================

def exact_sparse_row_match(
    A,
    B
):

    A = A.tocsr()
    B = B.tocsr()

    assert A.shape == B.shape

    diff = (
        A != B
    ).tocsr()

    total_diff_cells = (
        diff.nnz
    )

    row_diff_counts = np.diff(
        diff.indptr
    )

    row_matches = (
        row_diff_counts
        ==
        0
    )

    return {
        "rows": A.shape[0],
        "cols": A.shape[1],
        "matching_rows": int(
            row_matches.sum()
        ),
        "mismatching_rows": int(
            (~row_matches).sum()
        ),
        "match_rate": float(
            row_matches.mean()
        ),
        "different_cells": int(
            total_diff_cells
        ),
        "row_match_mask": row_matches
    }


# ============================================================
# 8) RECONSTRUCT TEST FEATURES
# ============================================================

print("\n" + "=" * 120)
print("RECONSTRUCTING TEST MORGAN FEATURES")
print("=" * 120)

X_test_rebuilt = reconstruct_pair_matrix(
    test_df
)

print(
    "Rebuilt test:",
    X_test_rebuilt.shape
)

print(
    "Saved X_test:",
    X_test.shape
)


test_alignment = exact_sparse_row_match(
    X_test_rebuilt,
    X_test
)


print("\n" + "=" * 120)
print("TEST ROW ALIGNMENT RESULT")
print("=" * 120)

for k, v in test_alignment.items():

    if k == "row_match_mask":
        continue

    print(
        f"{k:20s}:",
        v
    )


# ============================================================
# 9) RECONSTRUCT CALIBRATION FEATURES
# ============================================================

print("\n" + "=" * 120)
print("RECONSTRUCTING CALIBRATION MORGAN FEATURES")
print("=" * 120)

X_cal_rebuilt = reconstruct_pair_matrix(
    cal_df
)

cal_alignment = exact_sparse_row_match(
    X_cal_rebuilt,
    X_cal
)


print("\n" + "=" * 120)
print("CALIBRATION ROW ALIGNMENT RESULT")
print("=" * 120)

for k, v in cal_alignment.items():

    if k == "row_match_mask":
        continue

    print(
        f"{k:20s}:",
        v
    )


# ============================================================
# 10) HARD ALIGNMENT ASSERTION
# ============================================================

ALIGNMENT_CONFIRMED = (
    test_alignment[
        "mismatching_rows"
    ]
    ==
    0
    and
    cal_alignment[
        "mismatching_rows"
    ]
    ==
    0
)


print("\n" + "=" * 120)
print("FINAL ALIGNMENT DECISION")
print("=" * 120)

print(
    "ALIGNMENT_CONFIRMED =",
    ALIGNMENT_CONFIRMED
)


if not ALIGNMENT_CONFIRMED:

    print(
        "\n STOP."
    )

    print(
        "Interaction split order and Side-Effect "
        "feature order are NOT identical."
    )

    print(
        "Do not calculate integrated metrics."
    )

else:

    print(
        "\n Exact Morgan row identity confirmed."
    )

    print(
        " Interaction test row i corresponds "
        "to Side-Effect prediction row i."
    )

    print(
        " Ordered Drug1→Drug2 orientation confirmed."
    )


# ============================================================
# 11) CROSS-MATCH AGAINST FINAL SIDE-EFFECT DATASET
# ============================================================

# We use:
# Drug1 SMILES
# Drug2 SMILES
# exact Interaction text (Label)
#
# We do NOT modify any IDs or labels.

print("\n" + "=" * 120)
print("MATCH SPLITS TO FINAL SIDE-EFFECT DATASET")
print("=" * 120)


merge_cols = [
    "Drug1",
    "Drug2",
    "Label"
]


assert all(
    c in side_full.columns
    for c in merge_cols
)


side_lookup = (
    side_full[
        [
            "Drug1_ID",
            "Drug1",
            "Drug2_ID",
            "Drug2",
            "Label",
            "Interaction",
            "sideeffect_str"
        ]
    ]
    .copy()
)


print(
    "Duplicate keys in full side dataset:",
    int(
        side_lookup.duplicated(
            merge_cols
        ).sum()
    )
)


def attach_side_metadata(
    split_df,
    split_name
):

    x = (
        split_df
        .reset_index(
            names="split_row"
        )
        .merge(
            side_lookup,
            on=merge_cols,
            how="left",
            validate="one_to_one"
        )
    )

    matched = (
        x[
            "Interaction"
        ]
        .notna()
        .sum()
    )

    print(
        f"{split_name:6s}: "
        f"{matched:,}/{len(x):,} "
        f"rows matched"
    )

    return x


train_meta = attach_side_metadata(
    train_df,
    "TRAIN"
)

cal_meta = attach_side_metadata(
    cal_df,
    "CAL"
)

test_meta = attach_side_metadata(
    test_df,
    "TEST"
)


assert train_meta[
    "Interaction"
].notna().all()

assert cal_meta[
    "Interaction"
].notna().all()

assert test_meta[
    "Interaction"
].notna().all()


# ============================================================
# 12) CHECK label_id ↔ ORIGINAL Interaction ID RELATION
# ============================================================

print("\n" + "=" * 120)
print("LABEL-ID / INTERACTION-ID AUDIT")
print("=" * 120)


mapping_check = (

    pd.concat(
        [
            train_meta,
            cal_meta,
            test_meta
        ],
        ignore_index=True
    )

    .groupby(
        [
            "label_id",
            "Interaction",
            "Label"
        ]
    )

    .size()

    .reset_index(
        name="Rows"
    )
)


print(
    "Unique learned label_id:",
    mapping_check[
        "label_id"
    ].nunique()
)

print(
    "Unique preserved Interaction IDs:",
    mapping_check[
        "Interaction"
    ].nunique()
)

print(
    "Unique exact Interaction Labels:",
    mapping_check[
        "Label"
    ].nunique()
)


print(
    "\nFirst 20 mappings:"
)

print(
    mapping_check[
        [
            "label_id",
            "Interaction",
            "Rows",
            "Label"
        ]
    ]
    .sort_values(
        "label_id"
    )
    .head(
        20
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 13) SIDE-EFFECT DEPENDENCE ON INTERACTION CLASS
# ============================================================

print("\n" + "=" * 120)
print("SIDE-EFFECT TARGET DEPENDENCE AUDIT")
print("=" * 120)


class_side_map = (

    pd.concat(
        [
            train_meta,
            cal_meta,
            test_meta
        ],
        ignore_index=True
    )

    .groupby(
        [
            "Interaction",
            "Label"
        ],
        as_index=False
    )

    .agg(
        Rows=(
            "sideeffect_str",
            "size"
        ),

        Unique_SideEffect_Lists=(
            "sideeffect_str",
            "nunique"
        )
    )
)


print(
    "Interaction classes:",
    len(
        class_side_map
    )
)


print(
    "Classes with exactly ONE sideeffect list:",
    int(
        (
            class_side_map[
                "Unique_SideEffect_Lists"
            ]
            ==
            1
        ).sum()
    )
)


print(
    "Classes with >1 sideeffect list:",
    int(
        (
            class_side_map[
                "Unique_SideEffect_Lists"
            ]
            >
            1
        ).sum()
    )
)


print(
    "\nDistribution:"
)

print(
    class_side_map[
        "Unique_SideEffect_Lists"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 14) CHECKPOINT AUDIT
# ============================================================

print("\n" + "=" * 120)
print("BEST CHECKPOINT AUDIT")
print("=" * 120)


try:

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False
    )

except TypeError:

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu"
    )


print(
    "Checkpoint object type:",
    type(
        checkpoint
    )
)


if isinstance(
    checkpoint,
    dict
):

    print(
        "\nTop-level checkpoint keys:"
    )

    for key in checkpoint.keys():
        print(
            " •",
            key
        )


    print(
        "\nCheckpoint value summary:"
    )

    for key, value in checkpoint.items():

        if torch.is_tensor(
            value
        ):

            print(
                f"{key:30s}: "
                f"Tensor {tuple(value.shape)} "
                f"{value.dtype}"
            )

        elif isinstance(
            value,
            dict
        ):

            print(
                f"{key:30s}: "
                f"dict ({len(value)} keys)"
            )

        elif isinstance(
            value,
            (
                list,
                tuple
            )
        ):

            preview = value[:5]

            print(
                f"{key:30s}: "
                f"{type(value).__name__} "
                f"len={len(value)} "
                f"preview={preview}"
            )

        else:

            print(
                f"{key:30s}: "
                f"{type(value).__name__} "
                f"= {value}"
            )


else:

    print(
        "\nCheckpoint is not a dict."
    )

    print(
        checkpoint
    )


# ============================================================
# 15) FIND STATE DICT
# ============================================================

print("\n" + "=" * 120)
print("MODEL STATE DICT SHAPES")
print("=" * 120)


state_dict = None
state_key = None


if isinstance(
    checkpoint,
    dict
):

    candidate_keys = [
        "model_state_dict",
        "state_dict",
        "model",
        "net",
        "model_state",
        "weights"
    ]


    for key in candidate_keys:

        if key not in checkpoint:
            continue

        value = checkpoint[
            key
        ]


        if isinstance(
            value,
            dict
        ):

            if any(
                torch.is_tensor(v)
                for v in value.values()
            ):

                state_dict = value
                state_key = key
                break


    if state_dict is None:

        # maybe checkpoint itself is state_dict
        if all(
            isinstance(k, str)
            for k in checkpoint.keys()
        ) and any(
            torch.is_tensor(v)
            for v in checkpoint.values()
        ):

            state_dict = checkpoint
            state_key = "CHECKPOINT_ITSELF"


if state_dict is not None:

    print(
        "State dict source:",
        state_key
    )

    print(
        "Number of parameters/buffers:",
        len(
            state_dict
        )
    )

    for name, tensor in state_dict.items():

        if torch.is_tensor(
            tensor
        ):

            print(
                f"{name:60s} "
                f"{tuple(tensor.shape)}"
            )

else:

    print(
        " Could not automatically identify a state_dict."
    )


# ============================================================
# 16) CHECK FOR SAVED PREDICTIONS INSIDE CHECKPOINT
# ============================================================

print("\n" + "=" * 120)
print("CHECKPOINT PREDICTION-LIKE OBJECTS")
print("=" * 120)


prediction_tokens = [
    "pred",
    "prob",
    "logit",
    "test",
    "output"
]


found_prediction_object = False


if isinstance(
    checkpoint,
    dict
):

    for key, value in checkpoint.items():

        if any(
            token in key.lower()
            for token in prediction_tokens
        ):

            found_prediction_object = True

            print(
                "Candidate key:",
                key,
                "| type:",
                type(
                    value
                )
            )

            if torch.is_tensor(
                value
            ):

                print(
                    " shape:",
                    tuple(
                        value.shape
                    )
                )

            elif isinstance(
                value,
                np.ndarray
            ):

                print(
                    " shape:",
                    value.shape
                )


if not found_prediction_object:

    print(
        "No obvious saved test predictions "
        "inside checkpoint."
    )


# ============================================================
# 17) FEATURE CACHE AUDIT
# ============================================================

print("\n" + "=" * 120)
print("EXACT V2 FEATURE CACHE AUDIT")
print("=" * 120)


if FEATURE_CACHE_PATH.exists():

    cache = np.load(
        FEATURE_CACHE_PATH,
        allow_pickle=True
    )

    print(
        "Cache keys:"
    )

    for key in cache.files:

        arr = cache[
            key
        ]

        try:

            print(
                f" • {key:30s} "
                f"shape={arr.shape}, "
                f"dtype={arr.dtype}"
            )

        except Exception:

            print(
                " •",
                key,
                type(
                    arr
                )
            )

else:

    print(
        "Feature cache missing."
    )


# ============================================================
# 18) FEATURE META AUDIT
# ============================================================

print("\n" + "=" * 120)
print("EXACT V2 FEATURE META")
print("=" * 120)


if FEATURE_META_PATH.exists():

    with open(
        FEATURE_META_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        feature_meta = json.load(
            f
        )

    print(
        json.dumps(
            feature_meta,
            indent=2,
            ensure_ascii=False
        )[:20000]
    )

else:

    print(
        "Feature meta missing."
    )


# ============================================================
# 19) INTERACTION LABEL MAPPING
# ============================================================

print("\n" + "=" * 120)
print("INTERACTION LABEL MAPPING FILE")
print("=" * 120)


if INTERACTION_LABEL_MAP.exists():

    int_map = pd.read_csv(
        INTERACTION_LABEL_MAP
    )

    print(
        "Shape:",
        int_map.shape
    )

    print(
        "Columns:",
        int_map.columns.tolist()
    )

    print(
        int_map.head(
            20
        ).to_string(
            index=False
        )
    )

else:

    print(
        "Interaction label mapping missing."
    )


# ============================================================
# 20) SAVE ALIGNMENT METADATA
# ============================================================

OUT = Path(
    "/kaggle/working/"
    "INTEGRATED_FINAL_ALIGNMENT_AUDIT"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


test_meta.to_csv(
    OUT /
    "aligned_test_rows.csv",
    index=False
)


cal_meta.to_csv(
    OUT /
    "aligned_calibration_rows.csv",
    index=False
)


train_meta.to_csv(
    OUT /
    "aligned_train_rows.csv",
    index=False
)


class_side_map.to_csv(
    OUT /
    "interaction_sideeffect_dependence.csv",
    index=False
)


mapping_check.to_csv(
    OUT /
    "learned_label_to_preserved_interaction_id.csv",
    index=False
)


alignment_summary = {
    "test_rows": int(
        len(
            test_df
        )
    ),

    "calibration_rows": int(
        len(
            cal_df
        )
    ),

    "test_exact_morgan_matching_rows": int(
        test_alignment[
            "matching_rows"
        ]
    ),

    "test_mismatching_rows": int(
        test_alignment[
            "mismatching_rows"
        ]
    ),

    "cal_exact_morgan_matching_rows": int(
        cal_alignment[
            "matching_rows"
        ]
    ),

    "cal_mismatching_rows": int(
        cal_alignment[
            "mismatching_rows"
        ]
    ),

    "alignment_confirmed": bool(
        ALIGNMENT_CONFIRMED
    ),

    "train_test_ordered_pair_overlap": int(
        len(
            train_test_overlap
        )
    ),

    "train_cal_ordered_pair_overlap": int(
        len(
            train_cal_overlap
        )
    ),

    "cal_test_ordered_pair_overlap": int(
        len(
            cal_test_overlap
        )
    ),

    "interaction_classes": int(
        class_side_map.shape[
            0
        ]
    ),

    "classes_with_one_sideeffect_list": int(
        (
            class_side_map[
                "Unique_SideEffect_Lists"
            ]
            ==
            1
        ).sum()
    )
}


with open(
    OUT /
    "alignment_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        alignment_summary,
        f,
        indent=2
    )


print("\n" + "=" * 120)
print("FINAL STEP-2B SUMMARY")
print("=" * 120)

print(
    json.dumps(
        alignment_summary,
        indent=2
    )
)

print(
    "\nSaved to:"
)

print(
    OUT
)

print(
    "\n NO TRAINING PERFORMED"
)

Source cell: 33.


In [ ]:
# ============================================================
# INTEGRATED FINAL — STEP 2E V2
# EXACT SIDE-EFFECT SPLIT RECOVERY BY REPRODUCING THE SPLIT
# ============================================================
#
# NO TRAINING
#
# Strategy:
# 1) Start from final 191,795 rows
# 2) Remove exactly the 10 molecular-invalid rows -> 191,785
# 3) Reproduce 70 / 15 / 15 Interaction-stratified split
# 4) Test plausible split seeds, starting with 42
# 5) Accept a split ONLY if saved X_test/X_cal AND Y_test/Y_cal
#    match row-by-row exactly
#
# ============================================================

from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.model_selection import train_test_split

from rdkit import Chem
from rdkit.Chem import AllChem

warnings.filterwarnings("ignore")


# ============================================================
# 1) PATHS
# ============================================================

INTERACTION_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-v2-menna-projectt"
)

SIDE_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

FULL_PATH = (
    SIDE_ROOT /
    "cleaned_ddi_sideeffects_84_263_UNIQUE.csv"
)

SIDE_LABEL_MAP_PATH = (
    SIDE_ROOT /
    "label_mapping.csv"
)

X_TEST_PATH = SIDE_ROOT / "X_test.npz"
X_CAL_PATH  = SIDE_ROOT / "X_calibration.npz"

Y_TEST_PATH = SIDE_ROOT / "Y_test.npy"
Y_CAL_PATH  = SIDE_ROOT / "Y_calibration.npy"

INT_TRAIN_PATH = (
    INTERACTION_ROOT /
    "clean_train_split.csv"
)

INT_CAL_PATH = (
    INTERACTION_ROOT /
    "clean_calibration_split.csv"
)

INT_TEST_PATH = (
    INTERACTION_ROOT /
    "clean_test_split.csv"
)

OUT = Path(
    "/kaggle/working/"
    "INTEGRATED_FINAL_SIDE_SPLIT_RECOVERY_V2"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2) LOAD
# ============================================================

full_df = pd.read_csv(
    FULL_PATH
)

label_map = (
    pd.read_csv(
        SIDE_LABEL_MAP_PATH
    )
    .sort_values("label_id")
    .reset_index(drop=True)
)

X_test = sp.load_npz(
    X_TEST_PATH
).tocsr()

X_cal = sp.load_npz(
    X_CAL_PATH
).tocsr()

Y_test = np.load(
    Y_TEST_PATH
)

Y_cal = np.load(
    Y_CAL_PATH
)

int_train = pd.read_csv(
    INT_TRAIN_PATH
)

int_cal = pd.read_csv(
    INT_CAL_PATH
)

int_test = pd.read_csv(
    INT_TEST_PATH
)


print("=" * 110)
print("INPUT SHAPES")
print("=" * 110)

print("Final full dataset:", full_df.shape)
print("X_cal :", X_cal.shape)
print("X_test:", X_test.shape)
print("Y_cal :", Y_cal.shape)
print("Y_test:", Y_test.shape)

assert full_df.shape[0] == 191795
assert X_cal.shape == (28768, 4096)
assert X_test.shape == (28768, 4096)
assert Y_cal.shape == (28768, 263)
assert Y_test.shape == (28768, 263)
assert len(label_map) == 263


# ============================================================
# 3) FILTER THE 10 MOLECULAR-INVALID ROWS
# ============================================================

print("\n" + "=" * 110)
print("MOLECULAR VALIDITY")
print("=" * 110)

unique_smiles = pd.unique(
    pd.concat(
        [
            full_df["Drug1"],
            full_df["Drug2"]
        ],
        ignore_index=True
    )
)

smiles_valid = {}

for value in unique_smiles:

    if pd.isna(value):
        smiles_valid[value] = False

    else:
        smiles_valid[value] = (
            Chem.MolFromSmiles(
                str(value)
            )
            is not None
        )


def valid_smi(x):

    if pd.isna(x):
        return False

    return bool(
        smiles_valid.get(
            x,
            False
        )
    )


valid_mask = (
    full_df["Drug1"].map(valid_smi)
    &
    full_df["Drug2"].map(valid_smi)
)


invalid_df = (
    full_df.loc[
        ~valid_mask
    ]
    .copy()
)

valid_df = (
    full_df.loc[
        valid_mask
    ]
    .copy()
)


# keep row number from final 191,795 dataset
valid_df.insert(
    0,
    "full_dataset_row",
    valid_df.index.astype(int)
)

valid_df = valid_df.reset_index(
    drop=True
)


print("Full rows   :", len(full_df))
print("Invalid rows:", len(invalid_df))
print("Valid rows  :", len(valid_df))

assert len(invalid_df) == 10
assert len(valid_df) == 191785


invalid_df.to_csv(
    OUT /
    "excluded_10_molecular_invalid_rows.csv",
    index=False
)


# ============================================================
# 4) FINAL SIDE-EFFECT LABEL DICTIONARY
# ============================================================

label_to_id = dict(
    zip(
        label_map["label"].astype(str),
        label_map["label_id"].astype(int)
    )
)

assert len(label_to_id) == 263


def target_signature_from_string(value):

    terms = [
        x.strip()
        for x in str(value).split("|")
        if x.strip()
    ]

    missing = [
        x
        for x in terms
        if x not in label_to_id
    ]

    if missing:

        raise KeyError(
            f"Unknown side-effect labels: {missing}"
        )

    return tuple(
        sorted(
            label_to_id[x]
            for x in terms
        )
    )


full_target_sig = (
    valid_df["sideeffect_str"]
    .map(
        target_signature_from_string
    )
    .tolist()
)


def y_signatures(Y):

    return [
        tuple(
            np.flatnonzero(
                Y[i]
            ).astype(int).tolist()
        )
        for i in range(
            Y.shape[0]
        )
    ]


saved_cal_y_sig = y_signatures(
    Y_cal
)

saved_test_y_sig = y_signatures(
    Y_test
)


# ============================================================
# 5) EXACT MORGAN HASH FOR EACH UNIQUE DRUG
# ============================================================

print("\n" + "=" * 110)
print("BUILDING MORGAN HASH CACHE")
print("=" * 110)

all_valid_smiles = sorted(
    set(
        valid_df["Drug1"].astype(str)
    )
    |
    set(
        valid_df["Drug2"].astype(str)
    )
)

print(
    "Unique valid molecules:",
    len(all_valid_smiles)
)

fp_indices = {}


for n, smi in enumerate(
    all_valid_smiles,
    start=1
):

    mol = Chem.MolFromSmiles(
        smi
    )

    assert mol is not None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048,
        useChirality=False,
        useFeatures=False
    )

    fp_indices[smi] = np.asarray(
        fp.GetOnBits(),
        dtype=np.int32
    )

    if n % 500 == 0:

        print(
            f"processed "
            f"{n:,}/{len(all_valid_smiles):,}"
        )


def pair_hash(drug1, drug2):

    a = fp_indices[
        str(drug1)
    ]

    b = fp_indices[
        str(drug2)
    ] + 2048

    indices = np.concatenate(
        [a, b]
    ).astype(
        np.int32
    )

    values = np.ones(
        len(indices),
        dtype=np.int16
    )

    h = hashlib.sha1()

    h.update(
        indices.tobytes()
    )

    h.update(
        values.tobytes()
    )

    return h.hexdigest()


print("\nHashing full valid universe...")

full_pair_hash = []

for i, row in enumerate(
    valid_df.itertuples(index=False)
):

    full_pair_hash.append(
        pair_hash(
            row.Drug1,
            row.Drug2
        )
    )

    if (i + 1) % 25000 == 0:

        print(
            f"hashed "
            f"{i+1:,}/{len(valid_df):,}"
        )


# ============================================================
# 6) HASH SAVED X ROWS
# ============================================================

def sparse_hashes(X):

    X = X.tocsr(copy=True)
    X.sort_indices()

    hashes = []

    integer_like = np.allclose(
        X.data,
        np.round(
            X.data
        )
    )

    print(
        "integer-like:",
        integer_like
    )

    assert integer_like


    for i in range(
        X.shape[0]
    ):

        a = X.indptr[i]
        b = X.indptr[i + 1]

        indices = X.indices[
            a:b
        ].astype(
            np.int32,
            copy=False
        )

        values = np.round(
            X.data[a:b]
        ).astype(
            np.int16
        )

        h = hashlib.sha1()

        h.update(
            indices.tobytes()
        )

        h.update(
            values.tobytes()
        )

        hashes.append(
            h.hexdigest()
        )

    return hashes


print("\nHashing saved calibration X...")
saved_cal_hash = sparse_hashes(
    X_cal
)

print("Hashing saved test X...")
saved_test_hash = sparse_hashes(
    X_test
)


# ============================================================
# 7) HELPER — REPRODUCE ONE CANDIDATE SPLIT
# ============================================================

indices = np.arange(
    len(valid_df)
)

stratify_values = valid_df[
    "Interaction"
].astype(int).values


def reproduce_split(
    seed_first,
    seed_second
):

    train_idx, temp_idx = train_test_split(
        indices,
        test_size=0.30,
        random_state=seed_first,
        shuffle=True,
        stratify=stratify_values
    )


    temp_labels = stratify_values[
        temp_idx
    ]


    cal_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.50,
        random_state=seed_second,
        shuffle=True,
        stratify=temp_labels
    )


    return (
        np.asarray(
            train_idx,
            dtype=np.int64
        ),
        np.asarray(
            cal_idx,
            dtype=np.int64
        ),
        np.asarray(
            test_idx,
            dtype=np.int64
        )
    )


# ============================================================
# 8) FAST CANDIDATE CHECK
# ============================================================

def sequence_rate(
    indices_candidate,
    full_values,
    saved_values,
    first_n=None
):

    if first_n is not None:

        indices_candidate = (
            indices_candidate[
                :first_n
            ]
        )

        saved_values = (
            saved_values[
                :first_n
            ]
        )


    hits = sum(
        full_values[
            int(idx)
        ]
        ==
        observed

        for idx, observed in zip(
            indices_candidate,
            saved_values
        )
    )


    return (
        hits
        /
        len(
            indices_candidate
        )
    )


def candidate_score(
    seed1,
    seed2
):

    tr, ca, te = reproduce_split(
        seed1,
        seed2
    )


    # very fast prefix checks
    n = 100


    test_x_prefix = sequence_rate(
        te,
        full_pair_hash,
        saved_test_hash,
        first_n=n
    )


    cal_x_prefix = sequence_rate(
        ca,
        full_pair_hash,
        saved_cal_hash,
        first_n=n
    )


    test_y_prefix = sequence_rate(
        te,
        full_target_sig,
        saved_test_y_sig,
        first_n=n
    )


    cal_y_prefix = sequence_rate(
        ca,
        full_target_sig,
        saved_cal_y_sig,
        first_n=n
    )


    score = np.mean(
        [
            test_x_prefix,
            cal_x_prefix,
            test_y_prefix,
            cal_y_prefix
        ]
    )


    return {
        "seed_first":
            seed1,

        "seed_second":
            seed2,

        "test_X_prefix":
            test_x_prefix,

        "cal_X_prefix":
            cal_x_prefix,

        "test_Y_prefix":
            test_y_prefix,

        "cal_Y_prefix":
            cal_y_prefix,

        "mean_prefix_score":
            score
    }


# ============================================================
# 9) TRY THE EXPECTED / PLAUSIBLE SEEDS FIRST
# ============================================================

print("\n" + "=" * 110)
print("SEARCHING SPLIT SEED")
print("=" * 110)


candidate_pairs = []


# strongest historical candidate first
candidate_pairs.append(
    (42, 42)
)


# other seeds already used in experiments
for s in [
    43,
    44,
    0,
    1,
    7,
    11,
    17,
    21,
    41,
    123,
    2024,
    2025,
    2026
]:

    candidate_pairs.append(
        (s, s)
    )


# sometimes second split uses a separate fixed seed
for s1 in [
    42,
    43,
    44
]:

    for s2 in [
        42,
        43,
        44
    ]:

        candidate_pairs.append(
            (s1, s2)
        )


candidate_pairs = list(
    dict.fromkeys(
        candidate_pairs
    )
)


results = []


for s1, s2 in candidate_pairs:

    r = candidate_score(
        s1,
        s2
    )

    results.append(
        r
    )

    print(
        f"seed=({s1},{s2}) "
        f"score={r['mean_prefix_score']:.4f} "
        f"testX={r['test_X_prefix']:.4f} "
        f"calX={r['cal_X_prefix']:.4f} "
        f"testY={r['test_Y_prefix']:.4f} "
        f"calY={r['cal_Y_prefix']:.4f}"
    )


results_df = (
    pd.DataFrame(
        results
    )
    .sort_values(
        "mean_prefix_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("\nTop candidates:")
print(
    results_df.head(
        10
    ).to_string(
        index=False
    )
)


# ============================================================
# 10) IF NEEDED, SEARCH SEEDS 0..100
# ============================================================

best_prefix = float(
    results_df.iloc[0][
        "mean_prefix_score"
    ]
)


if best_prefix < 0.99:

    print(
        "\nNo exact obvious seed yet."
    )

    print(
        "Searching same-seed split over 0..100..."
    )


    already = set(
        candidate_pairs
    )


    for s in range(
        101
    ):

        if (
            s,
            s
        ) in already:

            continue


        r = candidate_score(
            s,
            s
        )

        results.append(
            r
        )


        if (
            r[
                "mean_prefix_score"
            ]
            >
            0.95
        ):

            print(
                "*** strong candidate:",
                r
            )


    results_df = (
        pd.DataFrame(
            results
        )
        .drop_duplicates(
            [
                "seed_first",
                "seed_second"
            ]
        )
        .sort_values(
            "mean_prefix_score",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


print("\n" + "=" * 110)
print("BEST PREFIX CANDIDATES")
print("=" * 110)

print(
    results_df.head(
        15
    ).to_string(
        index=False
    )
)


# ============================================================
# 11) FULL VERIFICATION OF BEST CANDIDATE
# ============================================================

best = results_df.iloc[
    0
]


BEST_SEED1 = int(
    best[
        "seed_first"
    ]
)

BEST_SEED2 = int(
    best[
        "seed_second"
    ]
)


train_idx, cal_idx, test_idx = (
    reproduce_split(
        BEST_SEED1,
        BEST_SEED2
    )
)


print("\n" + "=" * 110)
print("FULL VERIFICATION OF BEST CANDIDATE")
print("=" * 110)

print(
    "Seed first :",
    BEST_SEED1
)

print(
    "Seed second:",
    BEST_SEED2
)


test_x_hits = np.array(
    [
        full_pair_hash[
            int(idx)
        ]
        ==
        saved_test_hash[
            i
        ]

        for i, idx in enumerate(
            test_idx
        )
    ]
)


cal_x_hits = np.array(
    [
        full_pair_hash[
            int(idx)
        ]
        ==
        saved_cal_hash[
            i
        ]

        for i, idx in enumerate(
            cal_idx
        )
    ]
)


test_y_hits = np.array(
    [
        full_target_sig[
            int(idx)
        ]
        ==
        saved_test_y_sig[
            i
        ]

        for i, idx in enumerate(
            test_idx
        )
    ]
)


cal_y_hits = np.array(
    [
        full_target_sig[
            int(idx)
        ]
        ==
        saved_cal_y_sig[
            i
        ]

        for i, idx in enumerate(
            cal_idx
        )
    ]
)


print(
    "TEST X exact:",
    int(
        test_x_hits.sum()
    ),
    "/",
    len(
        test_x_hits
    )
)

print(
    "TEST Y exact:",
    int(
        test_y_hits.sum()
    ),
    "/",
    len(
        test_y_hits
    )
)

print(
    "CAL X exact :",
    int(
        cal_x_hits.sum()
    ),
    "/",
    len(
        cal_x_hits
    )
)

print(
    "CAL Y exact :",
    int(
        cal_y_hits.sum()
    ),
    "/",
    len(
        cal_y_hits
    )
)


EXACT_SPLIT_RECOVERED = bool(
    test_x_hits.all()
    and
    test_y_hits.all()
    and
    cal_x_hits.all()
    and
    cal_y_hits.all()
)


print(
    "\nEXACT_SPLIT_RECOVERED =",
    EXACT_SPLIT_RECOVERED
)


# ============================================================
# 12) IF EXACT, CREATE METADATA
# ============================================================

if not EXACT_SPLIT_RECOVERED:

    results_df.to_csv(
        OUT /
        "split_seed_search_results.csv",
        index=False
    )


    print(
        "\n Split was NOT accepted."
    )

    print(
        "Nothing was force-matched."
    )

    print(
        "Send BEST PREFIX CANDIDATES + FULL VERIFICATION output."
    )


else:

    print(
        "\n Exact historical SideEffect split reproduced."
    )


    side_train_meta = (
        valid_df.iloc[
            train_idx
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    side_cal_meta = (
        valid_df.iloc[
            cal_idx
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    side_test_meta = (
        valid_df.iloc[
            test_idx
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    side_train_meta.insert(
        0,
        "side_train_row",
        np.arange(
            len(
                side_train_meta
            )
        )
    )


    side_cal_meta.insert(
        0,
        "side_cal_row",
        np.arange(
            len(
                side_cal_meta
            )
        )
    )


    side_test_meta.insert(
        0,
        "side_test_row",
        np.arange(
            len(
                side_test_meta
            )
        )
    )


    side_train_meta.to_csv(
        OUT /
        "SIDE_EFFECT_TRAIN_METADATA_EXACT_ORDER.csv",
        index=False
    )


    side_cal_meta.to_csv(
        OUT /
        "SIDE_EFFECT_CAL_METADATA_EXACT_ORDER.csv",
        index=False
    )


    side_test_meta.to_csv(
        OUT /
        "SIDE_EFFECT_TEST_METADATA_EXACT_ORDER.csv",
        index=False
    )


    np.save(
        OUT /
        "side_train_full_positions.npy",
        train_idx
    )


    np.save(
        OUT /
        "side_cal_full_positions.npy",
        cal_idx
    )


    np.save(
        OUT /
        "side_test_full_positions.npy",
        test_idx
    )


    # ========================================================
    # 13) MAP INTERACTION SPLIT MEMBERSHIP
    # ========================================================

    KEY = [
        "Drug1",
        "Drug2",
        "Label"
    ]


    interaction_membership = pd.concat(
        [
            int_train[
                KEY
            ].assign(
                Interaction_Split="TRAIN"
            ),

            int_cal[
                KEY
            ].assign(
                Interaction_Split="CAL"
            ),

            int_test[
                KEY
            ].assign(
                Interaction_Split="TEST"
            )
        ],
        ignore_index=True
    )


    assert (
        interaction_membership
        .duplicated(
            KEY
        )
        .sum()
        ==
        0
    )


    side_test_with_int_split = (
        side_test_meta
        .merge(
            interaction_membership,
            on=KEY,
            how="left",
            validate="one_to_one"
        )
    )


    assert (
        side_test_with_int_split[
            "Interaction_Split"
        ]
        .notna()
        .all()
    )


    print("\n" + "=" * 110)
    print("SIDE TEST LOCATION IN INTERACTION SPLIT")
    print("=" * 110)


    print(
        side_test_with_int_split[
            "Interaction_Split"
        ]
        .value_counts()
        .to_string()
    )


    # ========================================================
    # 14) COMPLETE 3x3 CROSSTAB
    # ========================================================

    split_labels = np.full(
        len(
            valid_df
        ),
        "TRAIN",
        dtype=object
    )

    split_labels[
        cal_idx
    ] = "CAL"

    split_labels[
        test_idx
    ] = "TEST"


    full_membership = valid_df[
        KEY
    ].copy()


    full_membership[
        "SideEffect_Split"
    ] = split_labels


    full_membership = (
        full_membership
        .merge(
            interaction_membership,
            on=KEY,
            how="left",
            validate="one_to_one"
        )
    )


    cross_tab = pd.crosstab(
        full_membership[
            "SideEffect_Split"
        ],
        full_membership[
            "Interaction_Split"
        ],
        margins=True
    )


    print("\n" + "=" * 110)
    print("EXACT SIDE-EFFECT vs INTERACTION SPLIT CROSS-TAB")
    print("=" * 110)

    print(
        cross_tab.to_string()
    )


    cross_tab.to_csv(
        OUT /
        "EXACT_SIDE_VS_INTERACTION_SPLIT_CROSSTAB.csv"
    )


    side_test_with_int_split.to_csv(
        OUT /
        "SIDE_EFFECT_TEST_WITH_INTERACTION_SPLIT_MEMBERSHIP.csv",
        index=False
    )


    full_membership.to_csv(
        OUT /
        "FULL_191785_SPLIT_MEMBERSHIP.csv",
        index=False
    )


    # ========================================================
    # 15) OWN SIDE TRAIN/TEST PAIR OVERLAP
    # ========================================================

    train_pairs = set(
        zip(
            side_train_meta[
                "Drug1"
            ].astype(str),
            side_train_meta[
                "Drug2"
            ].astype(str)
        )
    )


    test_pairs = set(
        zip(
            side_test_meta[
                "Drug1"
            ].astype(str),
            side_test_meta[
                "Drug2"
            ].astype(str)
        )
    )


    overlap = (
        train_pairs
        &
        test_pairs
    )


    print("\n" + "=" * 110)
    print("SIDE-EFFECT OWN TRAIN/TEST ORDERED-PAIR OVERLAP")
    print("=" * 110)

    print(
        "Unique train pairs:",
        len(
            train_pairs
        )
    )

    print(
        "Unique test pairs :",
        len(
            test_pairs
        )
    )

    print(
        "Overlap pairs     :",
        len(
            overlap
        )
    )


    # ========================================================
    # 16) SAVE SUMMARY
    # ========================================================

    summary = {

        "exact_split_recovered":
            True,

        "first_split_seed":
            BEST_SEED1,

        "second_split_seed":
            BEST_SEED2,

        "valid_rows":
            int(
                len(
                    valid_df
                )
            ),

        "train_rows":
            int(
                len(
                    train_idx
                )
            ),

        "cal_rows":
            int(
                len(
                    cal_idx
                )
            ),

        "test_rows":
            int(
                len(
                    test_idx
                )
            ),

        "test_X_exact_rows":
            int(
                test_x_hits.sum()
            ),

        "test_Y_exact_rows":
            int(
                test_y_hits.sum()
            ),

        "cal_X_exact_rows":
            int(
                cal_x_hits.sum()
            ),

        "cal_Y_exact_rows":
            int(
                cal_y_hits.sum()
            ),

        "side_train_test_ordered_pair_overlap":
            int(
                len(
                    overlap
                )
            )
    }


    with open(
        OUT /
        "STEP2E_V2_SUMMARY.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            summary,
            f,
            indent=2
        )


    print("\n" + "=" * 110)
    print("FINAL STEP-2E V2 SUMMARY")
    print("=" * 110)

    print(
        json.dumps(
            summary,
            indent=2
        )
    )


print("\nSaved to:")
print(OUT)

print("\n NO TRAINING PERFORMED")

Source cell: 34.


In [ ]:
# ============================================================
# INTEGRATED FINAL — STEP 2D
# FINGERPRINT REPRESENTATION FORENSICS
# + AUTOMATIC ROW PERMUTATION RECOVERY
# ============================================================
#
# NO TRAINING
#
# Goal:
# Identify exactly how X_test/X_calibration were encoded:
# - Morgan bit vs Morgan count
# - radius 1 / 2 / 3
# - chirality False / True
# - atom features False / True
# - Drug1||Drug2 vs Drug2||Drug1
#
# If an exact representation is found:
# recover SideEffect row -> Interaction row permutation.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict, deque
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

warnings.filterwarnings("ignore")


# ============================================================
# 1) PATHS
# ============================================================

INTERACTION_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-v2-menna-projectt"
)

SIDE_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

TEST_PATH = (
    INTERACTION_ROOT /
    "clean_test_split.csv"
)

CAL_PATH = (
    INTERACTION_ROOT /
    "clean_calibration_split.csv"
)

SIDE_FULL_PATH = (
    SIDE_ROOT /
    "cleaned_ddi_sideeffects_84_263_UNIQUE.csv"
)

LABEL_MAP_PATH = (
    SIDE_ROOT /
    "label_mapping.csv"
)

X_TEST_PATH = (
    SIDE_ROOT /
    "X_test.npz"
)

X_CAL_PATH = (
    SIDE_ROOT /
    "X_calibration.npz"
)

Y_TEST_PATH = (
    SIDE_ROOT /
    "Y_test.npy"
)

Y_CAL_PATH = (
    SIDE_ROOT /
    "Y_calibration.npy"
)

OUT = Path(
    "/kaggle/working/"
    "INTEGRATED_FINAL_FP_FORENSICS"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2) LOAD
# ============================================================

test_df = pd.read_csv(
    TEST_PATH
)

cal_df = pd.read_csv(
    CAL_PATH
)

side_full = pd.read_csv(
    SIDE_FULL_PATH
)

label_map = (
    pd.read_csv(
        LABEL_MAP_PATH
    )
    .sort_values(
        "label_id"
    )
    .reset_index(
        drop=True
    )
)

X_test = sp.load_npz(
    X_TEST_PATH
).tocsr()

X_cal = sp.load_npz(
    X_CAL_PATH
).tocsr()

Y_test = np.load(
    Y_TEST_PATH
)

Y_cal = np.load(
    Y_CAL_PATH
)


assert X_test.shape == (28768, 4096)
assert X_cal.shape == (28768, 4096)
assert Y_test.shape == (28768, 263)
assert Y_cal.shape == (28768, 263)


# ============================================================
# 3) INSPECT SAVED X REPRESENTATION
# ============================================================

print("=" * 110)
print("SAVED X STATISTICS")
print("=" * 110)

for name, X in [
    ("TEST", X_test),
    ("CAL", X_cal)
]:

    data = X.data

    row_nnz = np.diff(
        X.indptr
    )

    print(f"\n{name}")

    print(
        "shape       :",
        X.shape
    )

    print(
        "dtype       :",
        X.dtype
    )

    print(
        "nnz         :",
        X.nnz
    )

    print(
        "density     :",
        X.nnz /
        (
            X.shape[0]
            *
            X.shape[1]
        )
    )

    print(
        "data min/max:",
        float(data.min()),
        float(data.max())
    )

    print(
        "first unique values:",
        np.unique(
            data
        )[:20]
    )

    print(
        "integer-like:",
        bool(
            np.allclose(
                data,
                np.round(data)
            )
        )
    )

    print(
        "row nnz mean:",
        float(
            row_nnz.mean()
        )
    )

    print(
        "row nnz min/max:",
        int(
            row_nnz.min()
        ),
        int(
            row_nnz.max()
        )
    )


# ============================================================
# 4) SPLIT SAVED PAIR FEATURES INTO 2048 + 2048
# ============================================================

X_test_A = X_test[:, :2048].tocsr()
X_test_B = X_test[:, 2048:].tocsr()

X_cal_A = X_cal[:, :2048].tocsr()
X_cal_B = X_cal[:, 2048:].tocsr()


# ============================================================
# 5) CANONICAL SPARSE ROW HASH
# ============================================================

def canonical_row_hashes(X):

    X = X.tocsr(copy=True)

    X.sort_indices()

    integer_like = np.allclose(
        X.data,
        np.round(X.data)
    )

    hashes = []

    for i in range(
        X.shape[0]
    ):

        start = X.indptr[i]
        end = X.indptr[i + 1]

        idx = X.indices[
            start:end
        ].astype(
            np.int32,
            copy=False
        )

        vals = X.data[
            start:end
        ]

        if integer_like:

            vals = np.round(
                vals
            ).astype(
                np.int16
            )

        else:

            vals = np.round(
                vals.astype(
                    np.float64
                ),
                7
            ).astype(
                np.float32
            )

        h = hashlib.sha1()

        h.update(
            idx.tobytes()
        )

        h.update(
            vals.tobytes()
        )

        hashes.append(
            h.hexdigest()
        )

    return hashes


print("\nHashing saved halves...")

test_A_hash = canonical_row_hashes(
    X_test_A
)

test_B_hash = canonical_row_hashes(
    X_test_B
)

cal_A_hash = canonical_row_hashes(
    X_cal_A
)

cal_B_hash = canonical_row_hashes(
    X_cal_B
)


# ============================================================
# 6) ALL UNIQUE DRUGS
# ============================================================

all_smiles = sorted(
    set(
        pd.concat(
            [
                test_df["Drug1"],
                test_df["Drug2"],
                cal_df["Drug1"],
                cal_df["Drug2"]
            ],
            ignore_index=True
        )
        .astype(str)
    )
)

print(
    "\nUnique drugs to test:",
    len(
        all_smiles
    )
)


mol_cache = {}

for smi in all_smiles:

    mol = Chem.MolFromSmiles(
        smi
    )

    if mol is None:

        raise ValueError(
            f"Invalid SMILES: {smi}"
        )

    mol_cache[
        smi
    ] = mol


# ============================================================
# 7) FINGERPRINT GENERATORS
# ============================================================

def make_fp(
    smiles,
    radius,
    chirality,
    use_features,
    mode
):

    mol = mol_cache[
        smiles
    ]

    if mode == "bit":

        bv = (
            AllChem
            .GetMorganFingerprintAsBitVect(
                mol,
                radius=radius,
                nBits=2048,
                useChirality=chirality,
                useFeatures=use_features
            )
        )

        arr = np.zeros(
            2048,
            dtype=np.int16
        )

        DataStructs.ConvertToNumpyArray(
            bv,
            arr
        )

        return arr


    elif mode == "count":

        fp = (
            AllChem
            .GetHashedMorganFingerprint(
                mol,
                radius=radius,
                nBits=2048,
                useChirality=chirality,
                useFeatures=use_features
            )
        )

        arr = np.zeros(
            2048,
            dtype=np.int16
        )

        for idx, value in (
            fp.GetNonzeroElements()
            .items()
        ):

            arr[
                int(idx)
            ] = int(value)

        return arr


    else:

        raise ValueError(
            mode
        )


def dense_hash(arr):

    arr = np.asarray(
        arr
    )

    idx = np.flatnonzero(
        arr
    ).astype(
        np.int32
    )

    vals = arr[
        idx
    ].astype(
        np.int16
    )

    h = hashlib.sha1()

    h.update(
        idx.tobytes()
    )

    h.update(
        vals.tobytes()
    )

    return h.hexdigest()


# ============================================================
# 8) MULTISET OVERLAP
# ============================================================

def multiset_rate(
    expected,
    observed
):

    a = Counter(
        expected
    )

    b = Counter(
        observed
    )

    overlap = sum(
        (
            a & b
        ).values()
    )

    return (
        overlap,
        overlap /
        len(expected),
        a == b
    )


# ============================================================
# 9) CANDIDATE REPRESENTATIONS
# ============================================================

candidate_specs = []

for mode in [
    "bit",
    "count"
]:

    for radius in [
        1,
        2,
        3
    ]:

        for chirality in [
            False,
            True
        ]:

            for use_features in [
                False,
                True
            ]:

                candidate_specs.append(
                    {
                        "mode":
                            mode,

                        "radius":
                            radius,

                        "chirality":
                            chirality,

                        "use_features":
                            use_features
                    }
                )


print(
    "\nCandidate representations:",
    len(
        candidate_specs
    )
)


# ============================================================
# 10) BUILD DRUG HASHES FOR EACH CANDIDATE
# ============================================================

results = []

candidate_hash_maps = {}


for n, spec in enumerate(
    candidate_specs,
    start=1
):

    name = (
        f"{spec['mode']}"
        f"_r{spec['radius']}"
        f"_ch{int(spec['chirality'])}"
        f"_feat{int(spec['use_features'])}"
    )

    print(
        f"\n[{n:02d}/{len(candidate_specs)}] "
        f"{name}"
    )

    drug_hash = {}

    for smi in all_smiles:

        arr = make_fp(
            smi,
            radius=spec[
                "radius"
            ],
            chirality=spec[
                "chirality"
            ],
            use_features=spec[
                "use_features"
            ],
            mode=spec[
                "mode"
            ]
        )

        drug_hash[
            smi
        ] = dense_hash(
            arr
        )


    candidate_hash_maps[
        name
    ] = drug_hash


    # TEST expected D1/D2 hashes
    expected_test_d1 = [
        drug_hash[s]
        for s
        in test_df[
            "Drug1"
        ].astype(str)
    ]

    expected_test_d2 = [
        drug_hash[s]
        for s
        in test_df[
            "Drug2"
        ].astype(str)
    ]


    # CAL expected
    expected_cal_d1 = [
        drug_hash[s]
        for s
        in cal_df[
            "Drug1"
        ].astype(str)
    ]

    expected_cal_d2 = [
        drug_hash[s]
        for s
        in cal_df[
            "Drug2"
        ].astype(str)
    ]


    # orientation A = D1, B = D2
    t_a_d1 = multiset_rate(
        expected_test_d1,
        test_A_hash
    )

    t_b_d2 = multiset_rate(
        expected_test_d2,
        test_B_hash
    )

    c_a_d1 = multiset_rate(
        expected_cal_d1,
        cal_A_hash
    )

    c_b_d2 = multiset_rate(
        expected_cal_d2,
        cal_B_hash
    )


    # reversed orientation
    t_a_d2 = multiset_rate(
        expected_test_d2,
        test_A_hash
    )

    t_b_d1 = multiset_rate(
        expected_test_d1,
        test_B_hash
    )


    direct_score = (
        t_a_d1[1]
        +
        t_b_d2[1]
        +
        c_a_d1[1]
        +
        c_b_d2[1]
    ) / 4


    reverse_score = (
        t_a_d2[1]
        +
        t_b_d1[1]
    ) / 2


    results.append(
        {
            "Candidate":
                name,

            "Mode":
                spec[
                    "mode"
                ],

            "Radius":
                spec[
                    "radius"
                ],

            "Chirality":
                spec[
                    "chirality"
                ],

            "UseFeatures":
                spec[
                    "use_features"
                ],

            "Test_A_as_Drug1":
                t_a_d1[
                    1
                ],

            "Test_B_as_Drug2":
                t_b_d2[
                    1
                ],

            "Cal_A_as_Drug1":
                c_a_d1[
                    1
                ],

            "Cal_B_as_Drug2":
                c_b_d2[
                    1
                ],

            "Direct_Mean":
                direct_score,

            "Test_A_as_Drug2":
                t_a_d2[
                    1
                ],

            "Test_B_as_Drug1":
                t_b_d1[
                    1
                ],

            "Reverse_Mean":
                reverse_score
        }
    )


# ============================================================
# 11) RESULTS TABLE
# ============================================================

results_df = (
    pd.DataFrame(
        results
    )
    .sort_values(
        "Direct_Mean",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 110)
print("CANDIDATE MATCH TABLE")
print("=" * 110)

print(
    results_df.head(
        24
    ).to_string(
        index=False
    )
)


results_df.to_csv(
    OUT /
    "fingerprint_candidate_match_table.csv",
    index=False
)


# ============================================================
# 12) BEST REPRESENTATION
# ============================================================

best = results_df.iloc[
    0
]

best_name = str(
    best[
        "Candidate"
    ]
)

best_direct = float(
    best[
        "Direct_Mean"
    ]
)

best_reverse = float(
    best[
        "Reverse_Mean"
    ]
)


print("\n" + "=" * 110)
print("BEST REPRESENTATION")
print("=" * 110)

print(
    best.to_string()
)


if (
    best_reverse
    >
    best_direct
):

    ORIENTATION = (
        "Drug2||Drug1"
    )

else:

    ORIENTATION = (
        "Drug1||Drug2"
    )


print(
    "\nBest orientation:",
    ORIENTATION
)


# ============================================================
# 13) BUILD PAIR HASH MULTISET FOR BEST CANDIDATE
# ============================================================

drug_hash = candidate_hash_maps[
    best_name
]


def build_expected_pair_keys(
    df,
    orientation
):

    keys = []

    for row in df.itertuples(
        index=False
    ):

        d1 = drug_hash[
            str(
                row.Drug1
            )
        ]

        d2 = drug_hash[
            str(
                row.Drug2
            )
        ]

        if orientation == "Drug1||Drug2":

            keys.append(
                (
                    d1,
                    d2
                )
            )

        else:

            keys.append(
                (
                    d2,
                    d1
                )
            )

    return keys


if ORIENTATION == "Drug1||Drug2":

    saved_test_pair_keys = list(
        zip(
            test_A_hash,
            test_B_hash
        )
    )

    saved_cal_pair_keys = list(
        zip(
            cal_A_hash,
            cal_B_hash
        )
    )

else:

    saved_test_pair_keys = list(
        zip(
            test_B_hash,
            test_A_hash
        )
    )

    saved_cal_pair_keys = list(
        zip(
            cal_B_hash,
            cal_A_hash
        )
    )


expected_test_pair_keys = (
    build_expected_pair_keys(
        test_df,
        "Drug1||Drug2"
    )
)

expected_cal_pair_keys = (
    build_expected_pair_keys(
        cal_df,
        "Drug1||Drug2"
    )
)


test_pair_counter_expected = Counter(
    expected_test_pair_keys
)

test_pair_counter_saved = Counter(
    saved_test_pair_keys
)


cal_pair_counter_expected = Counter(
    expected_cal_pair_keys
)

cal_pair_counter_saved = Counter(
    saved_cal_pair_keys
)


test_pair_overlap = sum(
    (
        test_pair_counter_expected
        &
        test_pair_counter_saved
    ).values()
)

cal_pair_overlap = sum(
    (
        cal_pair_counter_expected
        &
        cal_pair_counter_saved
    ).values()
)


print("\n" + "=" * 110)
print("BEST CANDIDATE PAIR-LEVEL MATCH")
print("=" * 110)

print(
    "TEST:",
    test_pair_overlap,
    "/",
    len(test_df),
    "=",
    test_pair_overlap /
    len(test_df)
)

print(
    "TEST exact multiset:",
    test_pair_counter_expected
    ==
    test_pair_counter_saved
)

print()

print(
    "CAL:",
    cal_pair_overlap,
    "/",
    len(cal_df),
    "=",
    cal_pair_overlap /
    len(cal_df)
)

print(
    "CAL exact multiset:",
    cal_pair_counter_expected
    ==
    cal_pair_counter_saved
)


# ============================================================
# 14) BUILD SIDE-EFFECT TRUE SIGNATURE
# ============================================================

label_to_id = dict(
    zip(
        label_map[
            "label"
        ].astype(str),
        label_map[
            "label_id"
        ].astype(int)
    )
)


interaction_to_side = (
    side_full
    .groupby(
        "Label"
    )[
        "sideeffect_str"
    ]
    .first()
    .to_dict()
)


def class_target_signature(
    exact_label
):

    s = interaction_to_side[
        exact_label
    ]

    terms = [
        x.strip()
        for x in str(
            s
        ).split("|")
        if x.strip()
    ]

    return tuple(
        sorted(
            label_to_id[
                x
            ]
            for x in terms
        )
    )


def y_row_signatures(
    Y
):

    return [
        tuple(
            np.flatnonzero(
                Y[i]
            ).tolist()
        )
        for i in range(
            len(Y)
        )
    ]


expected_test_y = [
    class_target_signature(
        x
    )
    for x in test_df[
        "Label"
    ]
]


expected_cal_y = [
    class_target_signature(
        x
    )
    for x in cal_df[
        "Label"
    ]
]


saved_test_y = y_row_signatures(
    Y_test
)

saved_cal_y = y_row_signatures(
    Y_cal
)


# ============================================================
# 15) COMPOSITE PAIR + Y KEY
# ============================================================

expected_test_composite = list(
    zip(
        expected_test_pair_keys,
        expected_test_y
    )
)

saved_test_composite = list(
    zip(
        saved_test_pair_keys,
        saved_test_y
    )
)


expected_cal_composite = list(
    zip(
        expected_cal_pair_keys,
        expected_cal_y
    )
)

saved_cal_composite = list(
    zip(
        saved_cal_pair_keys,
        saved_cal_y
    )
)


test_comp_expected = Counter(
    expected_test_composite
)

test_comp_saved = Counter(
    saved_test_composite
)

cal_comp_expected = Counter(
    expected_cal_composite
)

cal_comp_saved = Counter(
    saved_cal_composite
)


TEST_EXACT = (
    test_comp_expected
    ==
    test_comp_saved
)

CAL_EXACT = (
    cal_comp_expected
    ==
    cal_comp_saved
)


print("\n" + "=" * 110)
print("COMPOSITE PAIR + TARGET MATCH")
print("=" * 110)

print(
    "TEST exact:",
    TEST_EXACT
)

print(
    "CAL exact:",
    CAL_EXACT
)


# ============================================================
# 16) RECOVER PERMUTATION IF EXACT
# ============================================================

def recover_permutation(
    expected_keys,
    saved_keys
):

    lookup = defaultdict(
        deque
    )

    for interaction_idx, key in enumerate(
        expected_keys
    ):

        lookup[
            key
        ].append(
            interaction_idx
        )


    side_to_interaction = np.full(
        len(
            saved_keys
        ),
        -1,
        dtype=np.int64
    )


    for side_idx, key in enumerate(
        saved_keys
    ):

        if (
            key in lookup
            and
            len(
                lookup[
                    key
                ]
            )
            >
            0
        ):

            side_to_interaction[
                side_idx
            ] = (
                lookup[
                    key
                ]
                .popleft()
            )


    return side_to_interaction


print("\n" + "=" * 110)
print("PERMUTATION RECOVERY")
print("=" * 110)


if TEST_EXACT:

    test_perm = recover_permutation(
        expected_test_composite,
        saved_test_composite
    )

    print(
        "TEST mapped:",
        int(
            (
                test_perm
                >=
                0
            ).sum()
        ),
        "/",
        len(
            test_perm
        )
    )

    assert (
        test_perm
        >=
        0
    ).all()


    np.save(
        OUT /
        "test_side_to_interaction_row.npy",
        test_perm
    )


    aligned_test = (
        test_df.iloc[
            test_perm
        ]
        .reset_index(
            names="interaction_test_row"
        )
    )


    aligned_test.insert(
        0,
        "side_test_row",
        np.arange(
            len(
                aligned_test
            )
        )
    )


    aligned_test.to_csv(
        OUT /
        "interaction_test_aligned_to_side_order.csv",
        index=False
    )


else:

    test_perm = None

    print(
        "TEST permutation not recovered."
    )


if CAL_EXACT:

    cal_perm = recover_permutation(
        expected_cal_composite,
        saved_cal_composite
    )

    print(
        "CAL mapped:",
        int(
            (
                cal_perm
                >=
                0
            ).sum()
        ),
        "/",
        len(
            cal_perm
        )
    )

    assert (
        cal_perm
        >=
        0
    ).all()


    np.save(
        OUT /
        "cal_side_to_interaction_row.npy",
        cal_perm
    )


else:

    cal_perm = None

    print(
        "CAL permutation not recovered."
    )


# ============================================================
# 17) SUMMARY
# ============================================================

summary = {
    "best_candidate":
        best_name,

    "best_direct_score":
        best_direct,

    "best_reverse_score":
        best_reverse,

    "orientation":
        ORIENTATION,

    "test_pair_overlap":
        int(
            test_pair_overlap
        ),

    "test_pair_rows":
        int(
            len(
                test_df
            )
        ),

    "cal_pair_overlap":
        int(
            cal_pair_overlap
        ),

    "cal_pair_rows":
        int(
            len(
                cal_df
            )
        ),

    "test_pair_exact_multiset":
        bool(
            test_pair_counter_expected
            ==
            test_pair_counter_saved
        ),

    "cal_pair_exact_multiset":
        bool(
            cal_pair_counter_expected
            ==
            cal_pair_counter_saved
        ),

    "test_composite_exact":
        bool(
            TEST_EXACT
        ),

    "cal_composite_exact":
        bool(
            CAL_EXACT
        ),

    "test_rows_recovered":
        (
            int(
                (
                    test_perm
                    >=
                    0
                ).sum()
            )
            if test_perm is not None
            else 0
        ),

    "cal_rows_recovered":
        (
            int(
                (
                    cal_perm
                    >=
                    0
                ).sum()
            )
            if cal_perm is not None
            else 0
        )
}


print("\n" + "=" * 110)
print("FINAL STEP-2D SUMMARY")
print("=" * 110)

print(
    json.dumps(
        summary,
        indent=2
    )
)


with open(
    OUT /
    "step2d_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print(
    "\nSaved to:"
)

print(
    OUT
)

print(
    "\n NO TRAINING PERFORMED"
)

Source cell: 35.


In [ ]:
# ============================================================
# INTEGRATED FINAL — STEP 2E
# RECOVER THE EXACT SIDE-EFFECT TRAIN/CAL/TEST ROW MEMBERSHIP
# FROM THE FULL 191,785 MOLECULAR-VALID UNIVERSE
# ============================================================
#
# NO TRAINING
#
# Outputs:
# - exact SideEffect test metadata in prediction order
# - exact SideEffect calibration metadata in prediction order
# - exact SideEffect train membership
# - exact cross-tab against Interaction split membership
#
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp

from rdkit import Chem
from rdkit.Chem import AllChem

warnings.filterwarnings("ignore")


# ============================================================
# 1) PATHS
# ============================================================

INTERACTION_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-v2-menna-projectt"
)

SIDE_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

FULL_PATH = (
    SIDE_ROOT /
    "cleaned_ddi_sideeffects_84_263_UNIQUE.csv"
)

LABEL_MAP_PATH = (
    SIDE_ROOT /
    "label_mapping.csv"
)

X_TEST_PATH = SIDE_ROOT / "X_test.npz"
X_CAL_PATH  = SIDE_ROOT / "X_calibration.npz"

Y_TEST_PATH = SIDE_ROOT / "Y_test.npy"
Y_CAL_PATH  = SIDE_ROOT / "Y_calibration.npy"

INT_TRAIN_PATH = (
    INTERACTION_ROOT /
    "clean_train_split.csv"
)

INT_CAL_PATH = (
    INTERACTION_ROOT /
    "clean_calibration_split.csv"
)

INT_TEST_PATH = (
    INTERACTION_ROOT /
    "clean_test_split.csv"
)

OUT = Path(
    "/kaggle/working/"
    "INTEGRATED_FINAL_SIDE_SPLIT_RECOVERY"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2) LOAD
# ============================================================

full_df = pd.read_csv(
    FULL_PATH
)

label_map = (
    pd.read_csv(
        LABEL_MAP_PATH
    )
    .sort_values(
        "label_id"
    )
    .reset_index(
        drop=True
    )
)

X_test = sp.load_npz(
    X_TEST_PATH
).tocsr()

X_cal = sp.load_npz(
    X_CAL_PATH
).tocsr()

Y_test = np.load(
    Y_TEST_PATH
)

Y_cal = np.load(
    Y_CAL_PATH
)


int_train = pd.read_csv(
    INT_TRAIN_PATH
)

int_cal = pd.read_csv(
    INT_CAL_PATH
)

int_test = pd.read_csv(
    INT_TEST_PATH
)


print("=" * 110)
print("INPUT")
print("=" * 110)

print("Full SideEffect dataset :", full_df.shape)
print("X_test                  :", X_test.shape)
print("Y_test                  :", Y_test.shape)
print("X_cal                   :", X_cal.shape)
print("Y_cal                   :", Y_cal.shape)

print("Interaction train       :", int_train.shape)
print("Interaction cal         :", int_cal.shape)
print("Interaction test        :", int_test.shape)

assert len(full_df) == 191795
assert X_test.shape == (28768, 4096)
assert X_cal.shape == (28768, 4096)
assert Y_test.shape == (28768, 263)
assert Y_cal.shape == (28768, 263)
assert len(label_map) == 263


# ============================================================
# 3) FINAL 263 LABEL VOCABULARY
# ============================================================

label_to_id = dict(
    zip(
        label_map[
            "label"
        ].astype(str),
        label_map[
            "label_id"
        ].astype(int)
    )
)

assert len(label_to_id) == 263


# ============================================================
# 4) MOLECULAR VALIDITY AUDIT
# ============================================================

print("\n" + "=" * 110)
print("MOLECULAR VALIDITY AUDIT")
print("=" * 110)


all_drugs = pd.concat(
    [
        full_df["Drug1"],
        full_df["Drug2"]
    ],
    ignore_index=True
).drop_duplicates()


validity = {}


for value in all_drugs:

    if pd.isna(value):

        validity[value] = False
        continue

    smi = str(value)

    mol = Chem.MolFromSmiles(
        smi
    )

    validity[value] = (
        mol is not None
    )


def is_valid_smiles_value(x):

    if pd.isna(x):
        return False

    return bool(
        validity.get(
            x,
            False
        )
    )


valid_mask = (
    full_df[
        "Drug1"
    ].map(
        is_valid_smiles_value
    )
    &
    full_df[
        "Drug2"
    ].map(
        is_valid_smiles_value
    )
)


invalid_rows = (
    full_df.loc[
        ~valid_mask
    ]
    .copy()
)


valid_df = (
    full_df.loc[
        valid_mask
    ]
    .copy()
)


# preserve original final-dataset row number
valid_df.insert(
    0,
    "full_row",
    valid_df.index.astype(int)
)


valid_df = valid_df.reset_index(
    drop=True
)


print(
    "Original rows :",
    len(full_df)
)

print(
    "Invalid rows  :",
    len(invalid_rows)
)

print(
    "Valid rows    :",
    len(valid_df)
)


assert len(invalid_rows) == 10
assert len(valid_df) == 191785


invalid_rows.to_csv(
    OUT /
    "excluded_10_molecular_invalid_rows.csv",
    index=False
)


# ============================================================
# 5) EXACT DATASET KEY QA
# ============================================================

EXACT_KEY_COLS = [
    "Drug1",
    "Drug2",
    "Label"
]


exact_duplicate_count = int(
    valid_df.duplicated(
        EXACT_KEY_COLS
    ).sum()
)


print("\nExact Drug1+Drug2+Label duplicates:")
print(
    exact_duplicate_count
)


assert exact_duplicate_count == 0


# ============================================================
# 6) MORGAN BIT FP CACHE
# radius 2
# 2048
# no chirality
# no feature invariants
# ============================================================

print("\n" + "=" * 110)
print("BUILDING UNIQUE-DRUG MORGAN CACHE")
print("=" * 110)


unique_valid_smiles = sorted(
    set(
        valid_df[
            "Drug1"
        ].astype(str)
    )
    |
    set(
        valid_df[
            "Drug2"
        ].astype(str)
    )
)


print(
    "Unique valid molecules:",
    len(
        unique_valid_smiles
    )
)


fp_onbits = {}


for n, smi in enumerate(
    unique_valid_smiles,
    start=1
):

    mol = Chem.MolFromSmiles(
        smi
    )

    assert mol is not None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048,
        useChirality=False,
        useFeatures=False
    )

    fp_onbits[
        smi
    ] = np.asarray(
        list(
            fp.GetOnBits()
        ),
        dtype=np.int32
    )


    if n % 500 == 0:

        print(
            f"  processed "
            f"{n:,}/{len(unique_valid_smiles):,}"
        )


print(
    " FP cache ready"
)


# ============================================================
# 7) HASH HELPERS
# ============================================================

# Must match the canonical hash previously used for saved CSR rows.
# It hashes:
#   active indices as int32
#   corresponding integer values as int16

def hash_indices_and_values(
    indices,
    values
):

    indices = np.asarray(
        indices,
        dtype=np.int32
    )

    values = np.asarray(
        values,
        dtype=np.int16
    )

    h = hashlib.sha1()

    h.update(
        indices.tobytes()
    )

    h.update(
        values.tobytes()
    )

    return h.hexdigest()


def pair_hash_from_smiles(
    drug1,
    drug2
):

    d1 = fp_onbits[
        str(
            drug1
        )
    ]

    d2 = fp_onbits[
        str(
            drug2
        )
    ]


    # second fingerprint occupies columns 2048:4096
    indices = np.concatenate(
        [
            d1,
            d2 + 2048
        ]
    ).astype(
        np.int32
    )


    values = np.ones(
        len(
            indices
        ),
        dtype=np.int16
    )


    return hash_indices_and_values(
        indices,
        values
    )


def sparse_pair_hashes(
    X
):

    X = X.tocsr(
        copy=True
    )

    X.sort_indices()

    integer_like = np.allclose(
        X.data,
        np.round(
            X.data
        )
    )


    print(
        "Saved X integer-like:",
        integer_like
    )


    hashes = []


    for i in range(
        X.shape[0]
    ):

        start = X.indptr[
            i
        ]

        end = X.indptr[
            i + 1
        ]


        indices = X.indices[
            start:end
        ].astype(
            np.int32,
            copy=False
        )


        values = X.data[
            start:end
        ]


        if integer_like:

            values = np.round(
                values
            ).astype(
                np.int16
            )

        else:

            raise RuntimeError(
                "Saved Morgan X is unexpectedly non-integer."
            )


        hashes.append(
            hash_indices_and_values(
                indices,
                values
            )
        )


    return hashes


# ============================================================
# 8) BUILD PAIR HASH FOR ALL 191,785 VALID ROWS
# ============================================================

print("\n" + "=" * 110)
print("HASHING ALL MOLECULAR-VALID DATASET ROWS")
print("=" * 110)


full_pair_hashes = []


for i, row in enumerate(
    valid_df.itertuples(
        index=False
    )
):

    full_pair_hashes.append(
        pair_hash_from_smiles(
            row.Drug1,
            row.Drug2
        )
    )


    if (
        (i + 1)
        %
        25000
        ==
        0
    ):

        print(
            f"  hashed "
            f"{i+1:,}/{len(valid_df):,}"
        )


valid_df[
    "_pair_hash"
] = full_pair_hashes


print(
    "\n Full pair hashes complete"
)


# ============================================================
# 9) SIDE-EFFECT TARGET SIGNATURE FOR FULL DATA
# ============================================================

print("\n" + "=" * 110)
print("BUILDING FULL TARGET SIGNATURES")
print("=" * 110)


def sideeffect_string_to_sig(
    value
):

    terms = [
        x.strip()
        for x in str(
            value
        ).split("|")
        if x.strip()
    ]


    missing = [
        x
        for x in terms
        if x not in label_to_id
    ]


    if missing:

        raise KeyError(
            f"Unknown final label(s): {missing}"
        )


    return tuple(
        sorted(
            label_to_id[
                x
            ]
            for x in terms
        )
    )


valid_df[
    "_target_sig"
] = valid_df[
    "sideeffect_str"
].map(
    sideeffect_string_to_sig
)


# ============================================================
# 10) SAVED SIDE TEST/CAL HASHES + Y SIGNATURES
# ============================================================

print("\n" + "=" * 110)
print("HASHING SAVED SIDE-EFFECT SPLITS")
print("=" * 110)


test_pair_hashes = sparse_pair_hashes(
    X_test
)

cal_pair_hashes = sparse_pair_hashes(
    X_cal
)


def y_row_signatures(
    Y
):

    return [
        tuple(
            np.flatnonzero(
                Y[i]
            ).astype(
                int
            ).tolist()
        )
        for i in range(
            Y.shape[0]
        )
    ]


test_y_sig = y_row_signatures(
    Y_test
)

cal_y_sig = y_row_signatures(
    Y_cal
)


test_keys = list(
    zip(
        test_pair_hashes,
        test_y_sig
    )
)


cal_keys = list(
    zip(
        cal_pair_hashes,
        cal_y_sig
    )
)


full_keys = list(
    zip(
        valid_df[
            "_pair_hash"
        ],
        valid_df[
            "_target_sig"
        ]
    )
)


# ============================================================
# 11) KEY UNIQUENESS AUDIT
# ============================================================

print("\n" + "=" * 110)
print("FULL-UNIVERSE COMPOSITE KEY AUDIT")
print("=" * 110)


full_counter = Counter(
    full_keys
)

test_counter = Counter(
    test_keys
)

cal_counter = Counter(
    cal_keys
)


duplicate_full_keys = {
    k: n
    for k, n in full_counter.items()
    if n > 1
}


print(
    "Full valid rows:",
    len(
        full_keys
    )
)

print(
    "Unique composite keys:",
    len(
        full_counter
    )
)

print(
    "Duplicate composite keys:",
    len(
        duplicate_full_keys
    )
)


# ============================================================
# 12) CHECK TEST/CAL ARE SUBSETS OF FULL UNIVERSE
# ============================================================

test_missing = (
    test_counter
    -
    full_counter
)

cal_missing = (
    cal_counter
    -
    full_counter
)


n_test_missing = sum(
    test_missing.values()
)

n_cal_missing = sum(
    cal_missing.values()
)


print(
    "\nTEST rows not represented in full universe:",
    n_test_missing
)

print(
    "CAL rows not represented in full universe:",
    n_cal_missing
)


assert n_test_missing == 0
assert n_cal_missing == 0


# ============================================================
# 13) SAFETY: REQUIRE UNIQUE COMPOSITE KEYS
# ============================================================

if len(
    duplicate_full_keys
) > 0:

    print(
        "\n Composite Morgan+target collisions found."
    )

    collision_rows = []


    key_to_positions = {}


    for idx, key in enumerate(
        full_keys
    ):

        if (
            full_counter[
                key
            ]
            >
            1
        ):

            key_to_positions.setdefault(
                key,
                []
            ).append(
                idx
            )


    for collision_id, (
        key,
        positions
    ) in enumerate(
        key_to_positions.items()
    ):

        tmp = valid_df.iloc[
            positions
        ].copy()

        tmp[
            "collision_group"
        ] = collision_id

        collision_rows.append(
            tmp
        )


    collision_df = pd.concat(
        collision_rows,
        ignore_index=True
    )


    collision_df.to_csv(
        OUT /
        "composite_key_collisions.csv",
        index=False
    )


    print(
        "Collision rows saved for inspection:"
    )

    print(
        OUT /
        "composite_key_collisions.csv"
    )


    raise RuntimeError(
        "Unsafe ambiguity detected. "
        "Do not recover split identity automatically."
    )


print(
    " Every molecular-valid row has a unique Morgan+target key"
)


# ============================================================
# 14) FULL KEY -> FULL ROW LOOKUP
# ============================================================

key_to_position = {
    key: i
    for i, key in enumerate(
        full_keys
    )
}


# ============================================================
# 15) RECOVER TEST AND CAL ROWS IN EXACT SAVED ORDER
# ============================================================

test_positions = np.asarray(
    [
        key_to_position[
            key
        ]
        for key in test_keys
    ],
    dtype=np.int64
)


cal_positions = np.asarray(
    [
        key_to_position[
            key
        ]
        for key in cal_keys
    ],
    dtype=np.int64
)


assert len(
    np.unique(
        test_positions
    )
) == 28768


assert len(
    np.unique(
        cal_positions
    )
) == 28768


test_set = set(
    test_positions.tolist()
)

cal_set = set(
    cal_positions.tolist()
)


test_cal_overlap = (
    test_set
    &
    cal_set
)


print("\n" + "=" * 110)
print("SIDE SPLIT RECOVERY")
print("=" * 110)

print(
    "Recovered TEST:",
    len(
        test_positions
    )
)

print(
    "Recovered CAL :",
    len(
        cal_positions
    )
)

print(
    "TEST ∩ CAL   :",
    len(
        test_cal_overlap
    )
)


assert len(
    test_cal_overlap
) == 0


# ============================================================
# 16) TRAIN = REMAINDER
# ============================================================

all_positions = set(
    range(
        len(
            valid_df
        )
    )
)


train_set = (
    all_positions
    -
    test_set
    -
    cal_set
)


train_positions = np.asarray(
    sorted(
        train_set
    ),
    dtype=np.int64
)


print(
    "Recovered TRAIN:",
    len(
        train_positions
    )
)


assert len(
    train_positions
) == 134249


assert (
    len(
        train_positions
    )
    +
    len(
        cal_positions
    )
    +
    len(
        test_positions
    )
    ==
    191785
)


# ============================================================
# 17) BUILD METADATA FILES
# ============================================================

side_test_meta = (
    valid_df.iloc[
        test_positions
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


side_cal_meta = (
    valid_df.iloc[
        cal_positions
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


side_train_meta = (
    valid_df.iloc[
        train_positions
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


side_test_meta.insert(
    0,
    "side_test_row",
    np.arange(
        len(
            side_test_meta
        )
    )
)


side_cal_meta.insert(
    0,
    "side_cal_row",
    np.arange(
        len(
            side_cal_meta
        )
    )
)


# remove temporary internal columns from public output
DROP_INTERNAL = [
    "_pair_hash",
    "_target_sig"
]


side_test_out = (
    side_test_meta
    .drop(
        columns=DROP_INTERNAL
    )
)

side_cal_out = (
    side_cal_meta
    .drop(
        columns=DROP_INTERNAL
    )
)

side_train_out = (
    side_train_meta
    .drop(
        columns=DROP_INTERNAL
    )
)


side_test_out.to_csv(
    OUT /
    "SIDE_EFFECT_TEST_METADATA_EXACT_ORDER.csv",
    index=False
)

side_cal_out.to_csv(
    OUT /
    "SIDE_EFFECT_CAL_METADATA_EXACT_ORDER.csv",
    index=False
)

side_train_out.to_csv(
    OUT /
    "SIDE_EFFECT_TRAIN_METADATA.csv",
    index=False
)


np.save(
    OUT /
    "side_test_to_full_valid_position.npy",
    test_positions
)

np.save(
    OUT /
    "side_cal_to_full_valid_position.npy",
    cal_positions
)


# ============================================================
# 18) ROW-BY-ROW FEATURE + TARGET VERIFICATION
# ============================================================

print("\n" + "=" * 110)
print("ROW-BY-ROW RECOVERY VERIFICATION")
print("=" * 110)


recovered_test_pair_hash = (
    side_test_meta[
        "_pair_hash"
    ].tolist()
)

recovered_test_y_sig = (
    side_test_meta[
        "_target_sig"
    ].tolist()
)


test_pair_match = np.asarray(
    [
        a == b
        for a, b in zip(
            recovered_test_pair_hash,
            test_pair_hashes
        )
    ]
)


test_target_match = np.asarray(
    [
        a == b
        for a, b in zip(
            recovered_test_y_sig,
            test_y_sig
        )
    ]
)


print(
    "TEST feature hashes correct:",
    int(
        test_pair_match.sum()
    ),
    "/",
    len(
        test_pair_match
    )
)

print(
    "TEST target signatures correct:",
    int(
        test_target_match.sum()
    ),
    "/",
    len(
        test_target_match
    )
)


assert test_pair_match.all()
assert test_target_match.all()


recovered_cal_pair_hash = (
    side_cal_meta[
        "_pair_hash"
    ].tolist()
)

recovered_cal_y_sig = (
    side_cal_meta[
        "_target_sig"
    ].tolist()
)


cal_pair_match = np.asarray(
    [
        a == b
        for a, b in zip(
            recovered_cal_pair_hash,
            cal_pair_hashes
        )
    ]
)


cal_target_match = np.asarray(
    [
        a == b
        for a, b in zip(
            recovered_cal_y_sig,
            cal_y_sig
        )
    ]
)


print(
    "CAL feature hashes correct :",
    int(
        cal_pair_match.sum()
    ),
    "/",
    len(
        cal_pair_match
    )
)

print(
    "CAL target signatures correct:",
    int(
        cal_target_match.sum()
    ),
    "/",
    len(
        cal_target_match
    )
)


assert cal_pair_match.all()
assert cal_target_match.all()


# ============================================================
# 19) ASSIGN SIDE SPLIT NAME TO FULL VALID DATA
# ============================================================

side_split = np.full(
    len(
        valid_df
    ),
    "TRAIN",
    dtype=object
)


side_split[
    cal_positions
] = "CAL"


side_split[
    test_positions
] = "TEST"


valid_df[
    "SideEffect_Split"
] = side_split


# ============================================================
# 20) MAP INTERACTION SPLIT MEMBERSHIP TO SAME FULL UNIVERSE
# ============================================================

print("\n" + "=" * 110)
print("MAPPING INTERACTION SPLITS TO SAME 191,785 ROW UNIVERSE")
print("=" * 110)


int_train2 = (
    int_train[
        EXACT_KEY_COLS
    ]
    .copy()
)

int_train2[
    "Interaction_Split"
] = "TRAIN"


int_cal2 = (
    int_cal[
        EXACT_KEY_COLS
    ]
    .copy()
)

int_cal2[
    "Interaction_Split"
] = "CAL"


int_test2 = (
    int_test[
        EXACT_KEY_COLS
    ]
    .copy()
)

int_test2[
    "Interaction_Split"
] = "TEST"


interaction_membership = pd.concat(
    [
        int_train2,
        int_cal2,
        int_test2
    ],
    ignore_index=True
)


print(
    "Interaction membership rows:",
    len(
        interaction_membership
    )
)

print(
    "Duplicate exact membership keys:",
    int(
        interaction_membership
        .duplicated(
            EXACT_KEY_COLS
        )
        .sum()
    )
)


assert len(
    interaction_membership
) == 191785


assert (
    interaction_membership
    .duplicated(
        EXACT_KEY_COLS
    )
    .sum()
    ==
    0
)


membership_audit = (
    valid_df[
        EXACT_KEY_COLS
        +
        [
            "SideEffect_Split"
        ]
    ]
    .merge(
        interaction_membership,
        on=EXACT_KEY_COLS,
        how="left",
        validate="one_to_one"
    )
)


print(
    "Rows matched to Interaction membership:",
    int(
        membership_audit[
            "Interaction_Split"
        ]
        .notna()
        .sum()
    ),
    "/",
    len(
        membership_audit
    )
)


assert (
    membership_audit[
        "Interaction_Split"
    ]
    .notna()
    .all()
)


# ============================================================
# 21) EXACT 3x3 SPLIT CROSS-TAB
# ============================================================

cross_tab = pd.crosstab(
    membership_audit[
        "SideEffect_Split"
    ],
    membership_audit[
        "Interaction_Split"
    ],
    margins=True
)


# enforce readable order
desired_rows = [
    "TRAIN",
    "CAL",
    "TEST",
    "All"
]

desired_cols = [
    "TRAIN",
    "CAL",
    "TEST",
    "All"
]


cross_tab = cross_tab.reindex(
    index=[
        x
        for x in desired_rows
        if x in cross_tab.index
    ],
    columns=[
        x
        for x in desired_cols
        if x in cross_tab.columns
    ]
)


print("\n" + "=" * 110)
print("EXACT SIDE-EFFECT vs INTERACTION SPLIT CROSS-TAB")
print("=" * 110)

print(
    cross_tab.to_string()
)


cross_tab.to_csv(
    OUT /
    "EXACT_SIDE_VS_INTERACTION_SPLIT_CROSSTAB.csv"
)


# ============================================================
# 22) CRITICAL TEST OVERLAP
# ============================================================

side_test_membership = (
    membership_audit[
        membership_audit[
            "SideEffect_Split"
        ]
        ==
        "TEST"
    ]
)


side_test_in_int_train = int(
    (
        side_test_membership[
            "Interaction_Split"
        ]
        ==
        "TRAIN"
    ).sum()
)


side_test_in_int_cal = int(
    (
        side_test_membership[
            "Interaction_Split"
        ]
        ==
        "CAL"
    ).sum()
)


side_test_in_int_test = int(
    (
        side_test_membership[
            "Interaction_Split"
        ]
        ==
        "TEST"
    ).sum()
)


print("\n" + "=" * 110)
print("SIDE-EFFECT TEST LOCATION INSIDE INTERACTION SPLIT")
print("=" * 110)


print(
    "Side TEST rows that are Interaction TRAIN:",
    side_test_in_int_train
)

print(
    "Side TEST rows that are Interaction CAL  :",
    side_test_in_int_cal
)

print(
    "Side TEST rows that are Interaction TEST :",
    side_test_in_int_test
)

print(
    "Total:",
    side_test_in_int_train
    +
    side_test_in_int_cal
    +
    side_test_in_int_test
)


assert (
    side_test_in_int_train
    +
    side_test_in_int_cal
    +
    side_test_in_int_test
    ==
    28768
)


# ============================================================
# 23) PAIR OVERLAP FOR SIDE-EFFECT TRAIN/TEST
# ============================================================

def ordered_pairs(
    df
):

    return set(
        zip(
            df[
                "Drug1"
            ].astype(str),
            df[
                "Drug2"
            ].astype(str)
        )
    )


side_train_pairs = ordered_pairs(
    side_train_out
)

side_test_pairs = ordered_pairs(
    side_test_out
)


side_train_test_pair_overlap = (
    side_train_pairs
    &
    side_test_pairs
)


print("\n" + "=" * 110)
print("SIDE-EFFECT OWN TRAIN-TEST ORDERED PAIR OVERLAP")
print("=" * 110)


print(
    "Unique Side train pairs:",
    len(
        side_train_pairs
    )
)

print(
    "Unique Side test pairs :",
    len(
        side_test_pairs
    )
)

print(
    "Overlap:",
    len(
        side_train_test_pair_overlap
    )
)


# ============================================================
# 24) SAVE FULL MEMBERSHIP AUDIT
# ============================================================

membership_audit.to_csv(
    OUT /
    "FULL_191785_SPLIT_MEMBERSHIP_AUDIT.csv",
    index=False
)


summary = {

    "raw_final_rows":
        int(
            len(
                full_df
            )
        ),

    "molecular_invalid_rows":
        int(
            len(
                invalid_rows
            )
        ),

    "molecular_valid_rows":
        int(
            len(
                valid_df
            )
        ),

    "side_train_rows":
        int(
            len(
                side_train_out
            )
        ),

    "side_cal_rows":
        int(
            len(
                side_cal_out
            )
        ),

    "side_test_rows":
        int(
            len(
                side_test_out
            )
        ),

    "side_test_feature_rows_verified":
        int(
            test_pair_match.sum()
        ),

    "side_test_target_rows_verified":
        int(
            test_target_match.sum()
        ),

    "side_cal_feature_rows_verified":
        int(
            cal_pair_match.sum()
        ),

    "side_cal_target_rows_verified":
        int(
            cal_target_match.sum()
        ),

    "side_test_is_interaction_train":
        side_test_in_int_train,

    "side_test_is_interaction_cal":
        side_test_in_int_cal,

    "side_test_is_interaction_test":
        side_test_in_int_test,

    "side_own_train_test_ordered_pair_overlap":
        int(
            len(
                side_train_test_pair_overlap
            )
        ),

    "direct_rowwise_alignment_between_existing_test_sets":
        False,

    "integrated_evaluation_requires_new_interaction_inference_on_side_test":
        True
}


with open(
    OUT /
    "STEP2E_SUMMARY.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ============================================================
# 25) FINAL PRINT
# ============================================================

print("\n" + "=" * 110)
print("FINAL STEP-2E SUMMARY")
print("=" * 110)

print(
    json.dumps(
        summary,
        indent=2
    )
)


print("\nFiles saved in:")
print(
    OUT
)

print(
    "\n Exact SideEffect test row identities recovered"
)

print(
    " Exact SideEffect calibration row identities recovered"
)

print(
    " SideEffect train membership recovered by complement"
)

print(
    " Interaction vs SideEffect split relationship audited"
)

print(
    " NO TRAINING PERFORMED"
)

Source cell: 36.


In [ ]:
# ============================================================
# INTEGRATED FINAL — STEP 2F
# AMBIGUITY-SAFE SIDE-EFFECT TEST RECOVERY
# ============================================================
#
# NO TRAINING
#
# Purpose:
# 1) Map every saved SideEffect X/Y row to ALL possible rows
#    in the 191,785 molecular-valid universe.
# 2) Audit Morgan collisions.
# 3) Determine whether ambiguous rows are SAFE:
#       same Interaction ID
#       same exact Interaction Label
# 4) Recover canonical metadata for all 28,768 test rows.
#
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter, deque
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp

from rdkit import Chem
from rdkit.Chem import AllChem

warnings.filterwarnings("ignore")


# ============================================================
# 1) PATHS
# ============================================================

SIDE_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

FULL_PATH = (
    SIDE_ROOT /
    "cleaned_ddi_sideeffects_84_263_UNIQUE.csv"
)

LABEL_MAP_PATH = (
    SIDE_ROOT /
    "label_mapping.csv"
)

X_TEST_PATH = SIDE_ROOT / "X_test.npz"
X_CAL_PATH  = SIDE_ROOT / "X_calibration.npz"

Y_TEST_PATH = SIDE_ROOT / "Y_test.npy"
Y_CAL_PATH  = SIDE_ROOT / "Y_calibration.npy"

OUT = Path(
    "/kaggle/working/"
    "INTEGRATED_FINAL_AMBIGUITY_AUDIT"
)

if OUT.exists():
    import shutil
    shutil.rmtree(OUT)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2) LOAD
# ============================================================

full_df = pd.read_csv(
    FULL_PATH
)

label_map = (
    pd.read_csv(
        LABEL_MAP_PATH
    )
    .sort_values("label_id")
    .reset_index(drop=True)
)

X_test = sp.load_npz(
    X_TEST_PATH
).tocsr()

X_cal = sp.load_npz(
    X_CAL_PATH
).tocsr()

Y_test = np.load(
    Y_TEST_PATH
)

Y_cal = np.load(
    Y_CAL_PATH
)


print("=" * 110)
print("INPUT")
print("=" * 110)

print("Full rows:", len(full_df))
print("X_test   :", X_test.shape)
print("Y_test   :", Y_test.shape)
print("X_cal    :", X_cal.shape)
print("Y_cal    :", Y_cal.shape)

assert len(full_df) == 191795
assert X_test.shape == (28768, 4096)
assert Y_test.shape == (28768, 263)
assert X_cal.shape == (28768, 4096)
assert Y_cal.shape == (28768, 263)
assert len(label_map) == 263


# ============================================================
# 3) REMOVE THE 10 MOLECULAR-INVALID ROWS
# ============================================================

all_smiles = pd.unique(
    pd.concat(
        [
            full_df["Drug1"],
            full_df["Drug2"]
        ],
        ignore_index=True
    )
)

validity = {}

for x in all_smiles:

    if pd.isna(x):
        validity[x] = False
    else:
        validity[x] = (
            Chem.MolFromSmiles(
                str(x)
            )
            is not None
        )


def is_valid_smiles(x):

    if pd.isna(x):
        return False

    return bool(
        validity.get(
            x,
            False
        )
    )


valid_mask = (
    full_df["Drug1"].map(is_valid_smiles)
    &
    full_df["Drug2"].map(is_valid_smiles)
)


invalid_df = (
    full_df.loc[
        ~valid_mask
    ]
    .copy()
)


valid_df = (
    full_df.loc[
        valid_mask
    ]
    .copy()
)


valid_df.insert(
    0,
    "full_dataset_row",
    valid_df.index.astype(int)
)

valid_df = valid_df.reset_index(
    drop=True
)


print("\n" + "=" * 110)
print("MOLECULAR VALIDITY")
print("=" * 110)

print("Valid rows  :", len(valid_df))
print("Invalid rows:", len(invalid_df))

assert len(valid_df) == 191785
assert len(invalid_df) == 10


# ============================================================
# 4) LABEL VOCABULARY
# ============================================================

label_to_id = dict(
    zip(
        label_map["label"].astype(str),
        label_map["label_id"].astype(int)
    )
)

assert len(label_to_id) == 263


def target_signature(value):

    terms = [
        x.strip()
        for x in str(value).split("|")
        if x.strip()
    ]

    unknown = [
        x
        for x in terms
        if x not in label_to_id
    ]

    if unknown:
        raise KeyError(
            f"Unknown labels: {unknown}"
        )

    return tuple(
        sorted(
            label_to_id[x]
            for x in terms
        )
    )


valid_df["_target_sig"] = (
    valid_df["sideeffect_str"]
    .map(target_signature)
)


def y_signatures(Y):

    return [
        tuple(
            np.flatnonzero(
                Y[i]
            )
            .astype(int)
            .tolist()
        )
        for i in range(
            Y.shape[0]
        )
    ]


test_y_sig = y_signatures(
    Y_test
)

cal_y_sig = y_signatures(
    Y_cal
)


# ============================================================
# 5) BUILD EXACT MORGAN PAIR HASH FOR FULL UNIVERSE
# ============================================================

print("\n" + "=" * 110)
print("BUILDING MORGAN CACHE")
print("=" * 110)

unique_smiles = sorted(
    set(
        valid_df["Drug1"].astype(str)
    )
    |
    set(
        valid_df["Drug2"].astype(str)
    )
)

print(
    "Unique molecules:",
    len(unique_smiles)
)


fp_onbits = {}

for n, smi in enumerate(
    unique_smiles,
    start=1
):

    mol = Chem.MolFromSmiles(
        smi
    )

    assert mol is not None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048,
        useChirality=False,
        useFeatures=False
    )

    fp_onbits[smi] = np.asarray(
        fp.GetOnBits(),
        dtype=np.int32
    )

    if n % 500 == 0:
        print(
            f"processed "
            f"{n:,}/{len(unique_smiles):,}"
        )


def pair_hash_from_smiles(
    drug1,
    drug2
):

    a = fp_onbits[
        str(drug1)
    ]

    b = fp_onbits[
        str(drug2)
    ] + 2048

    idx = np.concatenate(
        [a, b]
    ).astype(
        np.int32
    )

    vals = np.ones(
        len(idx),
        dtype=np.int16
    )

    h = hashlib.sha1()

    h.update(
        idx.tobytes()
    )

    h.update(
        vals.tobytes()
    )

    return h.hexdigest()


print("\nHashing 191,785 full rows...")

full_hashes = []

for i, row in enumerate(
    valid_df.itertuples(index=False)
):

    full_hashes.append(
        pair_hash_from_smiles(
            row.Drug1,
            row.Drug2
        )
    )

    if (i + 1) % 25000 == 0:
        print(
            f"hashed "
            f"{i+1:,}/{len(valid_df):,}"
        )


valid_df["_pair_hash"] = full_hashes


# ============================================================
# 6) HASH SAVED X
# ============================================================

def csr_row_hashes(X):

    X = X.tocsr(copy=True)
    X.sort_indices()

    assert np.allclose(
        X.data,
        np.round(X.data)
    )

    hashes = []

    for i in range(
        X.shape[0]
    ):

        a = X.indptr[i]
        b = X.indptr[i + 1]

        idx = X.indices[
            a:b
        ].astype(
            np.int32,
            copy=False
        )

        vals = np.round(
            X.data[a:b]
        ).astype(
            np.int16
        )

        h = hashlib.sha1()

        h.update(
            idx.tobytes()
        )

        h.update(
            vals.tobytes()
        )

        hashes.append(
            h.hexdigest()
        )

    return hashes


print("\nHashing saved test...")
test_hashes = csr_row_hashes(
    X_test
)

print("Hashing saved calibration...")
cal_hashes = csr_row_hashes(
    X_cal
)


# ============================================================
# 7) COMPOSITE KEYS
# ============================================================

valid_df["_key"] = list(
    zip(
        valid_df["_pair_hash"],
        valid_df["_target_sig"]
    )
)


test_keys = list(
    zip(
        test_hashes,
        test_y_sig
    )
)

cal_keys = list(
    zip(
        cal_hashes,
        cal_y_sig
    )
)


# ============================================================
# 8) FULL KEY -> ALL CANDIDATE ROWS
# ============================================================

key_to_rows = defaultdict(list)

for i, key in enumerate(
    valid_df["_key"]
):

    key_to_rows[
        key
    ].append(i)


# ============================================================
# 9) VERIFY ALL SAVED ROWS EXIST
# ============================================================

missing_test = [
    i
    for i, key in enumerate(
        test_keys
    )
    if key not in key_to_rows
]

missing_cal = [
    i
    for i, key in enumerate(
        cal_keys
    )
    if key not in key_to_rows
]


print("\n" + "=" * 110)
print("COVERAGE")
print("=" * 110)

print(
    "TEST missing:",
    len(missing_test)
)

print(
    "CAL missing :",
    len(missing_cal)
)


assert len(missing_test) == 0
assert len(missing_cal) == 0


# ============================================================
# 10) COLLISION / AMBIGUITY AUDIT
# ============================================================

def audit_saved_keys(
    saved_keys,
    split_name
):

    rows = []

    for saved_row, key in enumerate(
        saved_keys
    ):

        candidates = key_to_rows[
            key
        ]

        candidate_df = (
            valid_df.iloc[
                candidates
            ]
        )

        interaction_ids = sorted(
            candidate_df[
                "Interaction"
            ]
            .astype(int)
            .unique()
            .tolist()
        )

        interaction_labels = sorted(
            candidate_df[
                "Label"
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        drug1_ids = sorted(
            candidate_df[
                "Drug1_ID"
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        drug2_ids = sorted(
            candidate_df[
                "Drug2_ID"
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        rows.append(
            {
                "Split":
                    split_name,

                "Saved_Row":
                    saved_row,

                "Candidate_Count":
                    len(candidates),

                "Interaction_ID_Count":
                    len(interaction_ids),

                "Interaction_Label_Count":
                    len(interaction_labels),

                "Interaction_IDs":
                    " | ".join(
                        map(
                            str,
                            interaction_ids
                        )
                    ),

                "Interaction_Labels":
                    " || ".join(
                        interaction_labels
                    ),

                "Drug1_ID_Count":
                    len(drug1_ids),

                "Drug2_ID_Count":
                    len(drug2_ids),

                "Safe_Interaction_Ambiguity":
                    (
                        len(interaction_ids) == 1
                        and
                        len(interaction_labels) == 1
                    )
            }
        )

    return pd.DataFrame(
        rows
    )


test_audit = audit_saved_keys(
    test_keys,
    "TEST"
)

cal_audit = audit_saved_keys(
    cal_keys,
    "CAL"
)


audit_all = pd.concat(
    [
        test_audit,
        cal_audit
    ],
    ignore_index=True
)


# ============================================================
# 11) SUMMARY OF AMBIGUITY
# ============================================================

print("\n" + "=" * 110)
print("TEST AMBIGUITY SUMMARY")
print("=" * 110)

print(
    "Rows with exactly 1 candidate:",
    int(
        (
            test_audit[
                "Candidate_Count"
            ]
            ==
            1
        ).sum()
    )
)

print(
    "Rows with >1 candidates:",
    int(
        (
            test_audit[
                "Candidate_Count"
            ]
            >
            1
        ).sum()
    )
)

print(
    "Rows with unsafe Interaction ambiguity:",
    int(
        (
            ~test_audit[
                "Safe_Interaction_Ambiguity"
            ]
        ).sum()
    )
)


print("\nCandidate-count distribution:")

print(
    test_audit[
        "Candidate_Count"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)


print("\n" + "=" * 110)
print("CAL AMBIGUITY SUMMARY")
print("=" * 110)

print(
    "Rows with exactly 1 candidate:",
    int(
        (
            cal_audit[
                "Candidate_Count"
            ]
            ==
            1
        ).sum()
    )
)

print(
    "Rows with >1 candidates:",
    int(
        (
            cal_audit[
                "Candidate_Count"
            ]
            >
            1
        ).sum()
    )
)

print(
    "Rows with unsafe Interaction ambiguity:",
    int(
        (
            ~cal_audit[
                "Safe_Interaction_Ambiguity"
            ]
        ).sum()
    )
)


# ============================================================
# 12) LIST UNSAFE CASES
# ============================================================

unsafe_test = (
    test_audit[
        ~test_audit[
            "Safe_Interaction_Ambiguity"
        ]
    ]
    .copy()
)

unsafe_cal = (
    cal_audit[
        ~cal_audit[
            "Safe_Interaction_Ambiguity"
        ]
    ]
    .copy()
)


if len(unsafe_test):

    print("\n" + "=" * 110)
    print("UNSAFE TEST CASES")
    print("=" * 110)

    print(
        unsafe_test.head(
            50
        ).to_string(
            index=False
        )
    )


if len(unsafe_cal):

    print("\n" + "=" * 110)
    print("UNSAFE CAL CASES")
    print("=" * 110)

    print(
        unsafe_cal.head(
            50
        ).to_string(
            index=False
        )
    )


# ============================================================
# 13) CANONICAL RECOVERY
# ============================================================

# This recovery is valid for Interaction evaluation ONLY
# if every candidate for a saved X+Y row has the same
# Interaction ID + exact Interaction Label.

TEST_SAFE = bool(
    test_audit[
        "Safe_Interaction_Ambiguity"
    ].all()
)

CAL_SAFE = bool(
    cal_audit[
        "Safe_Interaction_Ambiguity"
    ].all()
)


print("\n" + "=" * 110)
print("CANONICAL RECOVERY DECISION")
print("=" * 110)

print(
    "TEST_SAFE:",
    TEST_SAFE
)

print(
    "CAL_SAFE :",
    CAL_SAFE
)


# ============================================================
# 14) CREATE CANONICAL METADATA IF SAFE
# ============================================================

if TEST_SAFE:

    canonical_test_positions = []

    for key in test_keys:

        candidates = key_to_rows[
            key
        ]

        # deterministic representative
        canonical_test_positions.append(
            min(candidates)
        )


    test_meta = (
        valid_df.iloc[
            canonical_test_positions
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    test_meta.insert(
        0,
        "side_test_row",
        np.arange(
            len(
                test_meta
            )
        )
    )


    test_meta[
        "Candidate_Count"
    ] = test_audit[
        "Candidate_Count"
    ].values


    test_meta[
        "Was_Ambiguous_Morgan_Target_Key"
    ] = (
        test_meta[
            "Candidate_Count"
        ]
        >
        1
    )


    public_cols = [
        c
        for c in test_meta.columns
        if not c.startswith("_")
    ]


    test_meta[
        public_cols
    ].to_csv(
        OUT /
        "SIDE_EFFECT_TEST_CANONICAL_METADATA.csv",
        index=False
    )


    print(
        "\n TEST canonical metadata created:"
    )

    print(
        OUT /
        "SIDE_EFFECT_TEST_CANONICAL_METADATA.csv"
    )


if CAL_SAFE:

    canonical_cal_positions = []

    for key in cal_keys:

        candidates = key_to_rows[
            key
        ]

        canonical_cal_positions.append(
            min(candidates)
        )


    cal_meta = (
        valid_df.iloc[
            canonical_cal_positions
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    cal_meta.insert(
        0,
        "side_cal_row",
        np.arange(
            len(
                cal_meta
            )
        )
    )


    cal_meta[
        "Candidate_Count"
    ] = cal_audit[
        "Candidate_Count"
    ].values


    public_cols = [
        c
        for c in cal_meta.columns
        if not c.startswith("_")
    ]


    cal_meta[
        public_cols
    ].to_csv(
        OUT /
        "SIDE_EFFECT_CAL_CANONICAL_METADATA.csv",
        index=False
    )


# ============================================================
# 15) COLLISION DETAILS
# ============================================================

collision_detail_frames = []

collision_keys = set(
    key
    for key in test_keys
    if len(
        key_to_rows[
            key
        ]
    ) > 1
)


for collision_id, key in enumerate(
    collision_keys
):

    positions = key_to_rows[
        key
    ]

    tmp = (
        valid_df.iloc[
            positions
        ]
        .copy()
    )

    tmp.insert(
        0,
        "collision_group",
        collision_id
    )

    collision_detail_frames.append(
        tmp
    )


if collision_detail_frames:

    collision_details = pd.concat(
        collision_detail_frames,
        ignore_index=True
    )

    public_cols = [
        c
        for c in collision_details.columns
        if not c.startswith("_")
    ]

    collision_details[
        public_cols
    ].to_csv(
        OUT /
        "TEST_COLLISION_CANDIDATE_DETAILS.csv",
        index=False
    )

else:

    collision_details = pd.DataFrame()


# ============================================================
# 16) SAVE AUDIT TABLES
# ============================================================

test_audit.to_csv(
    OUT /
    "TEST_AMBIGUITY_AUDIT.csv",
    index=False
)

cal_audit.to_csv(
    OUT /
    "CAL_AMBIGUITY_AUDIT.csv",
    index=False
)

unsafe_test.to_csv(
    OUT /
    "UNSAFE_TEST_CASES.csv",
    index=False
)

unsafe_cal.to_csv(
    OUT /
    "UNSAFE_CAL_CASES.csv",
    index=False
)


# ============================================================
# 17) SUMMARY JSON
# ============================================================

summary = {

    "test_rows":
        int(
            len(test_keys)
        ),

    "cal_rows":
        int(
            len(cal_keys)
        ),

    "test_missing_keys":
        int(
            len(missing_test)
        ),

    "cal_missing_keys":
        int(
            len(missing_cal)
        ),

    "test_unique_candidate_rows":
        int(
            (
                test_audit[
                    "Candidate_Count"
                ]
                ==
                1
            ).sum()
        ),

    "test_ambiguous_candidate_rows":
        int(
            (
                test_audit[
                    "Candidate_Count"
                ]
                >
                1
            ).sum()
        ),

    "test_unsafe_interaction_rows":
        int(
            (
                ~test_audit[
                    "Safe_Interaction_Ambiguity"
                ]
            ).sum()
        ),

    "cal_unique_candidate_rows":
        int(
            (
                cal_audit[
                    "Candidate_Count"
                ]
                ==
                1
            ).sum()
        ),

    "cal_ambiguous_candidate_rows":
        int(
            (
                cal_audit[
                    "Candidate_Count"
                ]
                >
                1
            ).sum()
        ),

    "cal_unsafe_interaction_rows":
        int(
            (
                ~cal_audit[
                    "Safe_Interaction_Ambiguity"
                ]
            ).sum()
        ),

    "test_safe_for_interaction_label_evaluation":
        TEST_SAFE,

    "cal_safe_for_interaction_label_evaluation":
        CAL_SAFE
}


with open(
    OUT /
    "STEP2F_SUMMARY.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print("\n" + "=" * 110)
print("FINAL STEP-2F SUMMARY")
print("=" * 110)

print(
    json.dumps(
        summary,
        indent=2
    )
)


print("\nSaved to:")
print(OUT)

print("\n NO TRAINING PERFORMED")

Source cell: 37.


In [ ]:
# ============================================================
# INTEGRATED FINAL — STEP 2G
# EXACT INTERACTION CHECKPOINT INFERENCE RECIPE RECOVERY
# ============================================================
#
# NO TRAINING
#
# We recover the exact inference recipe by requiring the
# reconstructed model to reproduce the checkpoint's SAVED
# calibration metrics.
#
# ============================================================

from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

warnings.filterwarnings("ignore")


# ============================================================
# 0) PyG
# ============================================================

try:
    import torch_geometric
    from torch_geometric.nn import GCNConv

except Exception as e:
    raise RuntimeError(
        "torch_geometric is required for the Interaction checkpoint."
    ) from e


print("=" * 110)
print("ENVIRONMENT")
print("=" * 110)

print("Torch:", torch.__version__)
print("PyG  :", torch_geometric.__version__)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)


# ============================================================
# 1) PATHS
# ============================================================

ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-v2-menna-projectt"
)

TRAIN_PATH = ROOT / "clean_train_split.csv"
CAL_PATH   = ROOT / "clean_calibration_split.csv"
TEST_PATH  = ROOT / "clean_test_split.csv"

CACHE_PATH = (
    ROOT /
    "exact_v2_feature_cache_2249.npz"
)

CKPT_PATH = (
    ROOT /
    "BEST_checkpoint.pt"
)

OUT = Path(
    "/kaggle/working/"
    "INTEGRATED_FINAL_INTERACTION_RECIPE"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


for p in [
    TRAIN_PATH,
    CAL_PATH,
    TEST_PATH,
    CACHE_PATH,
    CKPT_PATH
]:
    assert p.exists(), p


# ============================================================
# 2) LOAD SPLITS
# ============================================================

train_df = pd.read_csv(
    TRAIN_PATH
)

cal_df = pd.read_csv(
    CAL_PATH
)

test_df = pd.read_csv(
    TEST_PATH
)


print("\n" + "=" * 110)
print("SPLITS")
print("=" * 110)

print("Train:", train_df.shape)
print("Cal  :", cal_df.shape)
print("Test :", test_df.shape)

print(
    "Train classes:",
    train_df["label_id"].nunique()
)

print(
    "Cal classes  :",
    cal_df["label_id"].nunique()
)

print(
    "Test classes :",
    test_df["label_id"].nunique()
)

assert train_df["label_id"].nunique() == 84
assert cal_df["label_id"].nunique() == 84
assert test_df["label_id"].nunique() == 84


# ============================================================
# 3) LOAD EXACT FEATURE CACHE
# ============================================================

cache = np.load(
    CACHE_PATH,
    allow_pickle=True
)

X_np = cache["x"].astype(
    np.float32
)

all_drugs = cache[
    "all_drugs"
].astype(str)


print("\n" + "=" * 110)
print("FEATURE CACHE")
print("=" * 110)

print("X:", X_np.shape)
print("Drugs:", all_drugs.shape)

assert X_np.shape == (1705, 2249)
assert len(all_drugs) == 1705
assert len(set(all_drugs)) == 1705


drug_to_idx = {
    smi: i
    for i, smi in enumerate(
        all_drugs
    )
}


# Every drug in splits must exist
split_drugs = set(
    pd.concat(
        [
            train_df["Drug1"],
            train_df["Drug2"],
            cal_df["Drug1"],
            cal_df["Drug2"],
            test_df["Drug1"],
            test_df["Drug2"]
        ],
        ignore_index=True
    ).astype(str)
)


missing_drugs = (
    split_drugs
    -
    set(
        all_drugs
    )
)

print(
    "Missing split drugs:",
    len(missing_drugs)
)

assert len(missing_drugs) == 0


X = torch.tensor(
    X_np,
    dtype=torch.float32,
    device=DEVICE
)


# ============================================================
# 4) LOAD CHECKPOINT
# ============================================================

try:

    ckpt = torch.load(
        CKPT_PATH,
        map_location="cpu",
        weights_only=False
    )

except TypeError:

    ckpt = torch.load(
        CKPT_PATH,
        map_location="cpu"
    )


print("\n" + "=" * 110)
print("CHECKPOINT")
print("=" * 110)

print("Seed        :", ckpt["seed"])
print("Best epoch  :", ckpt["best_epoch"])
print("Best macro  :", ckpt["best_cal_macro"])
print("Feature dim :", ckpt["feature_dim"])
print("Classes     :", ckpt["num_classes"])

assert ckpt["feature_dim"] == 2249
assert ckpt["num_classes"] == 84


# ============================================================
# 5) CHECK FILE CHECKSUMS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


print("\n" + "=" * 110)
print("CHECKSUM QA")
print("=" * 110)


for filename, expected in (
    ckpt.get(
        "input_checksums",
        {}
    ).items()
):

    if filename == "clean_label_names.npy":
        continue

    p = ROOT / filename

    if not p.exists():
        continue

    observed = sha256_file(
        p
    )

    print(
        filename,
        "->",
        observed == expected
    )

    assert observed == expected


# ============================================================
# 6) EXACT TARGET METRICS FROM CHECKPOINT HISTORY
# ============================================================

history = ckpt[
    "history"
]

best_epoch = int(
    ckpt[
        "best_epoch"
    ]
)


best_records = [
    r
    for r in history
    if int(
        r.get(
            "epoch",
            -1
        )
    )
    ==
    best_epoch
]


assert len(best_records) >= 1


best_record = best_records[
    -1
]


TARGET_ACC = float(
    best_record[
        "cal_accuracy"
    ]
)

TARGET_MACRO = float(
    best_record[
        "cal_macro_f1"
    ]
)

TARGET_WEIGHTED = float(
    best_record[
        "cal_weighted_f1"
    ]
)


print("\n" + "=" * 110)
print("CHECKPOINT CALIBRATION SIGNATURE")
print("=" * 110)

print(
    "Accuracy   :",
    TARGET_ACC
)

print(
    "Macro-F1   :",
    TARGET_MACRO
)

print(
    "Weighted-F1:",
    TARGET_WEIGHTED
)


assert abs(
    TARGET_MACRO
    -
    ckpt[
        "best_cal_macro"
    ]
) < 1e-10


# ============================================================
# 7) LABEL ORDER QA
# ============================================================

true_mapping = (
    pd.concat(
        [
            train_df[
                [
                    "label_id",
                    "Label"
                ]
            ],
            cal_df[
                [
                    "label_id",
                    "Label"
                ]
            ],
            test_df[
                [
                    "label_id",
                    "Label"
                ]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
    .sort_values(
        "label_id"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    true_mapping
) == 84


checkpoint_labels = list(
    map(
        str,
        ckpt[
            "label_names"
        ]
    )
)


split_labels = (
    true_mapping[
        "Label"
    ]
    .astype(str)
    .tolist()
)


print("\nCheckpoint label order exact:",
      checkpoint_labels == split_labels)

assert checkpoint_labels == split_labels


# ============================================================
# 8) PAIR INDICES
# ============================================================

def pair_indices(df):

    u = np.asarray(
        [
            drug_to_idx[
                str(x)
            ]
            for x in df[
                "Drug1"
            ]
        ],
        dtype=np.int64
    )

    v = np.asarray(
        [
            drug_to_idx[
                str(x)
            ]
            for x in df[
                "Drug2"
            ]
        ],
        dtype=np.int64
    )

    return u, v


train_u, train_v = pair_indices(
    train_df
)

cal_u, cal_v = pair_indices(
    cal_df
)

test_u, test_v = pair_indices(
    test_df
)


cal_pair_tensor = torch.tensor(
    np.column_stack(
        [
            cal_u,
            cal_v
        ]
    ),
    dtype=torch.long,
    device=DEVICE
)


test_pair_tensor = torch.tensor(
    np.column_stack(
        [
            test_u,
            test_v
        ]
    ),
    dtype=torch.long,
    device=DEVICE
)


y_cal = (
    cal_df[
        "label_id"
    ]
    .astype(int)
    .values
)

y_test = (
    test_df[
        "label_id"
    ]
    .astype(int)
    .values
)


# ============================================================
# 9) MODEL ARCHITECTURE
# ============================================================

def activation_function(
    x,
    name
):

    if name == "relu":
        return F.relu(x)

    if name == "elu":
        return F.elu(x)

    if name == "gelu":
        return F.gelu(x)

    if name == "leaky_relu":
        return F.leaky_relu(
            x,
            negative_slope=0.01
        )

    if name == "silu":
        return F.silu(x)

    raise ValueError(
        name
    )


def activation_module(
    name
):

    if name == "relu":
        return nn.ReLU()

    if name == "elu":
        return nn.ELU()

    if name == "gelu":
        return nn.GELU()

    if name == "leaky_relu":
        return nn.LeakyReLU(
            negative_slope=0.01
        )

    if name == "silu":
        return nn.SiLU()

    raise ValueError(
        name
    )


class ExactEncoder(nn.Module):

    def __init__(
        self,
        activation_name="relu",
        norm_order="bn_then_act",
        final_activation=True
    ):

        super().__init__()

        self.activation_name = (
            activation_name
        )

        self.norm_order = (
            norm_order
        )

        self.final_activation = (
            final_activation
        )


        self.conv1 = GCNConv(
            2249,
            384
        )

        self.bn1 = nn.BatchNorm1d(
            384
        )


        self.conv2 = GCNConv(
            384,
            384
        )

        self.bn2 = nn.BatchNorm1d(
            384
        )


        self.conv3 = GCNConv(
            384,
            192
        )

        self.bn3 = nn.BatchNorm1d(
            192
        )


    def block(
        self,
        x,
        conv,
        bn,
        edge_index,
        edge_weight,
        apply_activation=True
    ):

        x = conv(
            x,
            edge_index,
            edge_weight=edge_weight
        )


        if self.norm_order == "bn_then_act":

            x = bn(x)

            if apply_activation:

                x = activation_function(
                    x,
                    self.activation_name
                )


        elif self.norm_order == "act_then_bn":

            if apply_activation:

                x = activation_function(
                    x,
                    self.activation_name
                )

            x = bn(x)


        else:

            raise ValueError(
                self.norm_order
            )


        return x


    def forward(
        self,
        x,
        edge_index,
        edge_weight=None
    ):

        x = self.block(
            x,
            self.conv1,
            self.bn1,
            edge_index,
            edge_weight,
            apply_activation=True
        )


        x = self.block(
            x,
            self.conv2,
            self.bn2,
            edge_index,
            edge_weight,
            apply_activation=True
        )


        x = self.block(
            x,
            self.conv3,
            self.bn3,
            edge_index,
            edge_weight,
            apply_activation=self.final_activation
        )


        return x


class ExactInteractionModel(nn.Module):

    def __init__(
        self,
        activation_name="relu",
        norm_order="bn_then_act",
        final_activation=True
    ):

        super().__init__()


        self.encoder = ExactEncoder(
            activation_name=activation_name,
            norm_order=norm_order,
            final_activation=final_activation
        )


        act1 = activation_module(
            activation_name
        )

        act2 = activation_module(
            activation_name
        )


        # Module numbering intentionally matches checkpoint:
        #
        # classifier.0 = LayerNorm
        # classifier.1 = Linear 384 -> 512
        # classifier.4 = Linear 512 -> 256
        # classifier.7 = Linear 256 -> 84

        self.classifier = nn.Sequential(

            nn.LayerNorm(
                384
            ),                  # 0

            nn.Linear(
                384,
                512
            ),                  # 1

            act1,               # 2

            nn.Dropout(
                0.0
            ),                  # 3
                                # dropout disabled in eval anyway

            nn.Linear(
                512,
                256
            ),                  # 4

            act2,               # 5

            nn.Dropout(
                0.0
            ),                  # 6

            nn.Linear(
                256,
                84
            )                   # 7
        )


    def encode(
        self,
        x,
        edge_index,
        edge_weight=None
    ):

        return self.encoder(
            x,
            edge_index,
            edge_weight
        )


    def classify_pairs(
        self,
        h,
        pair_idx
    ):

        h1 = h[
            pair_idx[
                :,
                0
            ]
        ]

        h2 = h[
            pair_idx[
                :,
                1
            ]
        ]


        pair_repr = torch.cat(
            [
                h1,
                h2
            ],
            dim=1
        )


        return self.classifier(
            pair_repr
        )


# ============================================================
# 10) VERIFY STATE DICT CAN LOAD STRICTLY
# ============================================================

probe = ExactInteractionModel()

load_result = probe.load_state_dict(
    ckpt[
        "model_state"
    ],
    strict=True
)


print("\n" + "=" * 110)
print("STATE DICT QA")
print("=" * 110)

print(load_result)

print(
    " State dictionary matches reconstructed parameterized architecture."
)


del probe


# ============================================================
# 11) BUILD GRAPH CANDIDATES
# ============================================================

all_df = pd.concat(
    [
        train_df,
        cal_df,
        test_df
    ],
    ignore_index=True
)


def raw_edges(
    df
):

    u, v = pair_indices(
        df
    )

    return u, v


def make_graph(
    df,
    bidirectional=True,
    unique=False,
    edge_class_weight=False
):

    u, v = raw_edges(
        df
    )


    if edge_class_weight:

        assert "label_id" in df.columns

        labels = (
            df[
                "label_id"
            ]
            .astype(int)
            .values
        )

        counts = np.bincount(
            labels,
            minlength=84
        )

        weights = (
            len(labels)
            /
            (
                84
                *
                counts[
                    labels
                ]
            )
        ).astype(
            np.float32
        )

    else:

        weights = None


    if unique:

        pairs = np.column_stack(
            [
                u,
                v
            ]
        )

        if weights is None:

            pairs = np.unique(
                pairs,
                axis=0
            )

            u = pairs[:, 0]
            v = pairs[:, 1]

        else:

            # aggregate weights for repeated edges
            temp = pd.DataFrame(
                {
                    "u": u,
                    "v": v,
                    "w": weights
                }
            )

            temp = (
                temp
                .groupby(
                    [
                        "u",
                        "v"
                    ],
                    as_index=False
                )[
                    "w"
                ]
                .sum()
            )

            u = temp["u"].values
            v = temp["v"].values
            weights = temp["w"].values.astype(
                np.float32
            )


    if bidirectional:

        u0 = u.copy()
        v0 = v.copy()

        u = np.concatenate(
            [
                u0,
                v0
            ]
        )

        v = np.concatenate(
            [
                v0,
                u0
            ]
        )


        if weights is not None:

            weights = np.concatenate(
                [
                    weights,
                    weights
                ]
            )


    edge_index = torch.tensor(
        np.vstack(
            [
                u,
                v
            ]
        ),
        dtype=torch.long,
        device=DEVICE
    )


    if weights is None:

        edge_weight = None

    else:

        edge_weight = torch.tensor(
            weights,
            dtype=torch.float32,
            device=DEVICE
        )


    return (
        edge_index,
        edge_weight
    )


# ============================================================
# 12) METRIC EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_recipe(
    graph_df,
    bidirectional,
    unique,
    edge_class_weight,
    activation_name,
    norm_order,
    final_activation
):

    edge_index, edge_weight = make_graph(

        graph_df,

        bidirectional=
            bidirectional,

        unique=
            unique,

        edge_class_weight=
            edge_class_weight
    )


    model = ExactInteractionModel(

        activation_name=
            activation_name,

        norm_order=
            norm_order,

        final_activation=
            final_activation

    ).to(
        DEVICE
    )


    model.load_state_dict(
        ckpt[
            "model_state"
        ],
        strict=True
    )


    model.eval()


    h = model.encode(
        X,
        edge_index,
        edge_weight
    )


    logits = model.classify_pairs(
        h,
        cal_pair_tensor
    )


    pred = (
        logits
        .argmax(
            dim=1
        )
        .cpu()
        .numpy()
    )


    acc = accuracy_score(
        y_cal,
        pred
    )


    macro = f1_score(
        y_cal,
        pred,
        average="macro",
        zero_division=0
    )


    weighted = f1_score(
        y_cal,
        pred,
        average="weighted",
        zero_division=0
    )


    signature_error = (
        abs(
            acc
            -
            TARGET_ACC
        )
        +
        abs(
            macro
            -
            TARGET_MACRO
        )
        +
        abs(
            weighted
            -
            TARGET_WEIGHTED
        )
    )


    del model
    del h
    del logits

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


    return {
        "Accuracy":
            acc,

        "Macro_F1":
            macro,

        "Weighted_F1":
            weighted,

        "Signature_Error":
            signature_error,

        "Edges":
            int(
                edge_index.shape[
                    1
                ]
            )
    }


# ============================================================
# 13) STAGE A:
# MOST LIKELY GRAPH = TRAIN, BIDIRECTIONAL, ROW EDGES
# SEARCH PARAMETER-FREE ARCHITECTURE DETAILS
# ============================================================

print("\n" + "=" * 110)
print("STAGE A — ARCHITECTURE SEARCH")
print("=" * 110)


architecture_candidates = [

    # most likely first
    ("relu", "bn_then_act", True),
    ("relu", "bn_then_act", False),
    ("relu", "act_then_bn", True),
    ("relu", "act_then_bn", False),

    ("elu", "bn_then_act", True),
    ("elu", "bn_then_act", False),

    ("gelu", "bn_then_act", True),
    ("gelu", "bn_then_act", False),

    ("leaky_relu", "bn_then_act", True),
    ("leaky_relu", "bn_then_act", False),

    ("silu", "bn_then_act", True),
    ("silu", "bn_then_act", False),

    # only try alternate BN order if needed
    ("elu", "act_then_bn", True),
    ("gelu", "act_then_bn", True),
    ("leaky_relu", "act_then_bn", True),
    ("silu", "act_then_bn", True),
]


results = []


EXACT_TOL = 1e-10


for act, norm_order, final_act in (
    architecture_candidates
):

    r = evaluate_recipe(

        train_df,

        bidirectional=True,

        unique=False,

        edge_class_weight=False,

        activation_name=act,

        norm_order=norm_order,

        final_activation=final_act
    )


    row = {

        "Stage":
            "A",

        "Graph":
            "TRAIN_BIDIR_ROWS",

        "Activation":
            act,

        "Norm_Order":
            norm_order,

        "Final_Activation":
            final_act,

        "Edge_Class_Weight":
            False,

        **r
    }


    results.append(
        row
    )


    print(
        f"{act:12s} "
        f"{norm_order:12s} "
        f"final={str(final_act):5s} | "
        f"acc={r['Accuracy']:.10f} "
        f"macro={r['Macro_F1']:.10f} "
        f"weighted={r['Weighted_F1']:.10f} "
        f"error={r['Signature_Error']:.12g}"
    )


    if (
        r[
            "Signature_Error"
        ]
        <
        EXACT_TOL
    ):

        print(
            "\n EXACT SIGNATURE FOUND IN STAGE A"
        )

        break


results_df = pd.DataFrame(
    results
)


best_arch_row = (
    results_df
    .sort_values(
        "Signature_Error"
    )
    .iloc[0]
)


# ============================================================
# 14) STAGE B:
# IF NEEDED, SEARCH GRAPH CONSTRUCTION USING BEST ARCHITECTURE
# ============================================================

if float(
    best_arch_row[
        "Signature_Error"
    ]
) >= EXACT_TOL:

    print("\n" + "=" * 110)
    print("STAGE B — GRAPH CONSTRUCTION SEARCH")
    print("=" * 110)


    best_act = str(
        best_arch_row[
            "Activation"
        ]
    )

    best_norm = str(
        best_arch_row[
            "Norm_Order"
        ]
    )

    best_final = bool(
        best_arch_row[
            "Final_Activation"
        ]
    )


    graph_candidates = [

        (
            "TRAIN_DIRECT_ROWS",
            train_df,
            False,
            False,
            False
        ),

        (
            "TRAIN_BIDIR_UNIQUE",
            train_df,
            True,
            True,
            False
        ),

        (
            "TRAIN_DIRECT_UNIQUE",
            train_df,
            False,
            True,
            False
        ),

        (
            "ALL_BIDIR_ROWS",
            all_df,
            True,
            False,
            False
        ),

        (
            "ALL_DIRECT_ROWS",
            all_df,
            False,
            False,
            False
        ),

        (
            "ALL_BIDIR_UNIQUE",
            all_df,
            True,
            True,
            False
        ),

        (
            "ALL_DIRECT_UNIQUE",
            all_df,
            False,
            True,
            False
        ),

        # possible interpretation of "weighted graph"
        (
            "TRAIN_BIDIR_CLASSWEIGHT",
            train_df,
            True,
            False,
            True
        ),

        (
            "TRAIN_DIRECT_CLASSWEIGHT",
            train_df,
            False,
            False,
            True
        )
    ]


    for (
        graph_name,
        graph_df,
        bidir,
        unique,
        edge_cw
    ) in graph_candidates:


        r = evaluate_recipe(

            graph_df,

            bidirectional=
                bidir,

            unique=
                unique,

            edge_class_weight=
                edge_cw,

            activation_name=
                best_act,

            norm_order=
                best_norm,

            final_activation=
                best_final
        )


        row = {

            "Stage":
                "B",

            "Graph":
                graph_name,

            "Activation":
                best_act,

            "Norm_Order":
                best_norm,

            "Final_Activation":
                best_final,

            "Edge_Class_Weight":
                edge_cw,

            **r
        }


        results.append(
            row
        )


        print(
            f"{graph_name:28s} | "
            f"acc={r['Accuracy']:.10f} "
            f"macro={r['Macro_F1']:.10f} "
            f"weighted={r['Weighted_F1']:.10f} "
            f"error={r['Signature_Error']:.12g}"
        )


        if (
            r[
                "Signature_Error"
            ]
            <
            EXACT_TOL
        ):

            print(
                "\n EXACT SIGNATURE FOUND IN STAGE B"
            )

            break


# ============================================================
# 15) FINAL RESULT TABLE
# ============================================================

results_df = (
    pd.DataFrame(
        results
    )
    .sort_values(
        "Signature_Error"
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 110)
print("TOP RECONSTRUCTION CANDIDATES")
print("=" * 110)


print(
    results_df.head(
        15
    ).to_string(
        index=False
    )
)


results_df.to_csv(
    OUT /
    "interaction_recipe_search.csv",
    index=False
)


best = results_df.iloc[
    0
]


RECIPE_EXACT = bool(
    float(
        best[
            "Signature_Error"
        ]
    )
    <
    EXACT_TOL
)


print("\n" + "=" * 110)
print("BEST RECIPE")
print("=" * 110)

print(
    best.to_string()
)

print(
    "\nRECIPE_EXACT =",
    RECIPE_EXACT
)


# ============================================================
# 16) IF EXACT, REBUILD AND EVALUATE MAIN INTERACTION TEST
# ============================================================

if RECIPE_EXACT:

    graph_name = str(
        best[
            "Graph"
        ]
    )


    if graph_name.startswith(
        "TRAIN"
    ):

        graph_df = train_df

    else:

        graph_df = all_df


    bidirectional = (
        "BIDIR"
        in graph_name
    )


    unique = (
        "UNIQUE"
        in graph_name
    )


    edge_class_weight = bool(
        best[
            "Edge_Class_Weight"
        ]
    )


    edge_index, edge_weight = make_graph(

        graph_df,

        bidirectional=
            bidirectional,

        unique=
            unique,

        edge_class_weight=
            edge_class_weight
    )


    model = ExactInteractionModel(

        activation_name=
            str(
                best[
                    "Activation"
                ]
            ),

        norm_order=
            str(
                best[
                    "Norm_Order"
                ]
            ),

        final_activation=
            bool(
                best[
                    "Final_Activation"
                ]
            )

    ).to(
        DEVICE
    )


    model.load_state_dict(
        ckpt[
            "model_state"
        ],
        strict=True
    )


    model.eval()


    with torch.no_grad():

        h = model.encode(
            X,
            edge_index,
            edge_weight
        )


        test_logits = (
            model
            .classify_pairs(
                h,
                test_pair_tensor
            )
        )


        test_pred = (
            test_logits
            .argmax(
                dim=1
            )
            .cpu()
            .numpy()
        )


    test_acc = accuracy_score(
        y_test,
        test_pred
    )


    test_macro = f1_score(
        y_test,
        test_pred,
        average="macro",
        zero_division=0
    )


    test_weighted = f1_score(
        y_test,
        test_pred,
        average="weighted",
        zero_division=0
    )


    print("\n" + "=" * 110)
    print("MAIN INTERACTION TEST METRICS")
    print("=" * 110)

    print(
        f"Accuracy   : "
        f"{100*test_acc:.4f}%"
    )

    print(
        f"Macro-F1   : "
        f"{100*test_macro:.4f}%"
    )

    print(
        f"Weighted-F1: "
        f"{100*test_weighted:.4f}%"
    )


    np.save(
        OUT /
        "main_interaction_test_predictions.npy",
        test_pred
    )


    # Save node embeddings for next integrated step
    np.save(
        OUT /
        "exact_node_embeddings.npy",
        h.detach()
        .cpu()
        .numpy()
        .astype(
            np.float32
        )
    )


    recipe = {

        "recipe_exact":
            True,

        "graph":
            graph_name,

        "activation":
            str(
                best[
                    "Activation"
                ]
            ),

        "norm_order":
            str(
                best[
                    "Norm_Order"
                ]
            ),

        "final_activation":
            bool(
                best[
                    "Final_Activation"
                ]
            ),

        "edge_class_weight":
            edge_class_weight,

        "cal_accuracy":
            float(
                best[
                    "Accuracy"
                ]
            ),

        "cal_macro_f1":
            float(
                best[
                    "Macro_F1"
                ]
            ),

        "cal_weighted_f1":
            float(
                best[
                    "Weighted_F1"
                ]
            ),

        "main_test_accuracy":
            float(
                test_acc
            ),

        "main_test_macro_f1":
            float(
                test_macro
            ),

        "main_test_weighted_f1":
            float(
                test_weighted
            )
    }


    with open(
        OUT /
        "EXACT_INTERACTION_INFERENCE_RECIPE.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            recipe,
            f,
            indent=2
        )


    print(
        "\n Exact Interaction inference recipe recovered."
    )

    print(
        " Node embeddings saved for integrated evaluation."
    )

else:

    print(
        "\n No candidate reproduced the saved "
        "calibration signature exactly."
    )

    print(
        "Do NOT calculate integrated results yet."
    )


print("\nSaved to:")
print(OUT)

print("\n NO TRAINING PERFORMED")

Source cell: 39.


In [ ]:
# ============================================================
# INTEGRATED FINAL — STEP 2H
# LEAKAGE-SAFE COMMON HELD-OUT INTEGRATED EVALUATION
# ============================================================
#
# PRIMARY INTEGRATED TEST:
#   SideEffect TEST
#   ∩ exact molecular identity recovered
#   ∩ Interaction TEST
#
# This guarantees that every primary integrated row was held
# out from BOTH model-specific training/calibration splits.
#
# ALSO:
# - verifies original SideEffect primary ensemble signature
# - runs exact Interaction checkpoint classifier
# - computes:
#       Both correct
#       Interaction only correct
#       Side-effects only correct
#       Both wrong
#       Strict integrated accuracy
# - compares with train-only Interaction→SideEffect lookup
# - exports Excel / CSV / JSON / ZIP
#
# NO TRAINING
# ============================================================


from pathlib import Path
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss
)

warnings.filterwarnings("ignore")


# ============================================================
# 0) PATHS
# ============================================================

INT_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-v2-menna-projectt"
)

SIDE_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

STEP2F_ROOT = Path(
    "/kaggle/working/"
    "INTEGRATED_FINAL_AMBIGUITY_AUDIT"
)

STEP2G_ROOT = Path(
    "/kaggle/working/"
    "INTEGRATED_FINAL_INTERACTION_RECIPE"
)

BASELINE_ROOT = (
    SIDE_ROOT /
    "PRIMARY_UNIQUE263_BASELINE_PROBS"
)

# Interaction
INT_TRAIN_PATH = INT_ROOT / "clean_train_split.csv"
INT_CAL_PATH   = INT_ROOT / "clean_calibration_split.csv"
INT_TEST_PATH  = INT_ROOT / "clean_test_split.csv"

INT_TRAIN_ROWS = INT_ROOT / "train_rows.csv"

FEATURE_CACHE_PATH = (
    INT_ROOT /
    "exact_v2_feature_cache_2249.npz"
)

CHECKPOINT_PATH = (
    INT_ROOT /
    "BEST_checkpoint.pt"
)

# Step 2F
SIDE_TEST_META_PATH = (
    STEP2F_ROOT /
    "SIDE_EFFECT_TEST_CANONICAL_METADATA.csv"
)

TEST_AMBIG_AUDIT_PATH = (
    STEP2F_ROOT /
    "TEST_AMBIGUITY_AUDIT.csv"
)

COLLISION_DETAILS_PATH = (
    STEP2F_ROOT /
    "TEST_COLLISION_CANDIDATE_DETAILS.csv"
)

# Step 2G
NODE_EMBEDDINGS_PATH = (
    STEP2G_ROOT /
    "exact_node_embeddings.npy"
)

RECIPE_PATH = (
    STEP2G_ROOT /
    "EXACT_INTERACTION_INFERENCE_RECIPE.json"
)

# Side targets
Y_CAL_PATH = SIDE_ROOT / "Y_calibration.npy"
Y_TEST_PATH = SIDE_ROOT / "Y_test.npy"

SIDE_LABEL_MAP_PATH = (
    SIDE_ROOT /
    "label_mapping.csv"
)

OUT = Path(
    "/kaggle/working/"
    "FINAL_INTEGRATED_COMMON_HELDOUT_EVALUATION"
)

if OUT.exists():
    shutil.rmtree(OUT)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1) REQUIRED FILE QA
# ============================================================

required = {
    "Interaction train": INT_TRAIN_PATH,
    "Interaction cal": INT_CAL_PATH,
    "Interaction test": INT_TEST_PATH,
    "Interaction train rows": INT_TRAIN_ROWS,
    "Feature cache": FEATURE_CACHE_PATH,
    "BEST checkpoint": CHECKPOINT_PATH,
    "Step2F test metadata": SIDE_TEST_META_PATH,
    "Step2F ambiguity audit": TEST_AMBIG_AUDIT_PATH,
    "Step2G embeddings": NODE_EMBEDDINGS_PATH,
    "Step2G recipe": RECIPE_PATH,
    "Side Y_cal": Y_CAL_PATH,
    "Side Y_test": Y_TEST_PATH,
    "Side label map": SIDE_LABEL_MAP_PATH,
}

print("=" * 120)
print("PATH QA")
print("=" * 120)

for name, p in required.items():
    print(
        f"{name:26s}: "
        f"{p.exists()}  {p}"
    )
    assert p.exists(), p

for seed in [42, 43, 44]:

    assert (
        BASELINE_ROOT /
        f"seed_{seed}" /
        "calibration_probabilities.npy"
    ).exists()

    assert (
        BASELINE_ROOT /
        f"seed_{seed}" /
        "test_probabilities.npy"
    ).exists()


print("\n All required files found")


# ============================================================
# 2) LOAD INTERACTION SPLITS
# ============================================================

int_train = pd.read_csv(
    INT_TRAIN_PATH
)

int_cal = pd.read_csv(
    INT_CAL_PATH
)

int_test = pd.read_csv(
    INT_TEST_PATH
)

train_rows = pd.read_csv(
    INT_TRAIN_ROWS
)


print("\n" + "=" * 120)
print("INTERACTION SPLITS")
print("=" * 120)

print("Train:", int_train.shape)
print("Cal  :", int_cal.shape)
print("Test :", int_test.shape)

assert len(int_train) == 134249
assert len(int_cal) == 28768
assert len(int_test) == 28768

assert int_train["label_id"].nunique() == 84
assert int_cal["label_id"].nunique() == 84
assert int_test["label_id"].nunique() == 84


# ============================================================
# 3) INTERACTION LABEL MAPPING
# ============================================================

interaction_mapping = (
    pd.concat(
        [
            int_train[["label_id", "Label"]],
            int_cal[["label_id", "Label"]],
            int_test[["label_id", "Label"]]
        ],
        ignore_index=True
    )
    .drop_duplicates()
    .sort_values("label_id")
    .reset_index(drop=True)
)


assert len(interaction_mapping) == 84
assert interaction_mapping["label_id"].nunique() == 84
assert interaction_mapping["Label"].nunique() == 84


label_to_learned_id = dict(
    zip(
        interaction_mapping["Label"].astype(str),
        interaction_mapping["label_id"].astype(int)
    )
)

learned_id_to_label = dict(
    zip(
        interaction_mapping["label_id"].astype(int),
        interaction_mapping["Label"].astype(str)
    )
)


# ============================================================
# 4) SIDE-EFFECT TEST METADATA FROM STEP 2F
# ============================================================

side_meta = pd.read_csv(
    SIDE_TEST_META_PATH
)

ambiguity_audit = pd.read_csv(
    TEST_AMBIG_AUDIT_PATH
)


print("\n" + "=" * 120)
print("SIDE-EFFECT TEST RECOVERY")
print("=" * 120)

print("Side metadata:", side_meta.shape)
print("Ambiguity audit:", ambiguity_audit.shape)

assert len(side_meta) == 28768
assert len(ambiguity_audit) == 28768

assert np.array_equal(
    side_meta["side_test_row"].astype(int).values,
    np.arange(28768)
)

assert np.array_equal(
    ambiguity_audit["Saved_Row"].astype(int).values,
    np.arange(28768)
)


print(
    "Unique-identity rows:",
    int(
        (
            side_meta["Candidate_Count"] == 1
        ).sum()
    )
)

print(
    "Morgan-equivalent ambiguous rows:",
    int(
        (
            side_meta["Candidate_Count"] > 1
        ).sum()
    )
)

assert (
    side_meta["Candidate_Count"] == 1
).sum() == 27345

assert (
    side_meta["Candidate_Count"] > 1
).sum() == 1423


# ============================================================
# 5) MAP EVERY EXACT ROW TO ITS INTERACTION SPLIT
# ============================================================

KEY = [
    "Drug1",
    "Drug2",
    "Label"
]

interaction_membership = pd.concat(
    [
        int_train[
            KEY
        ].assign(
            Interaction_Split="TRAIN"
        ),

        int_cal[
            KEY
        ].assign(
            Interaction_Split="CAL"
        ),

        int_test[
            KEY
        ].assign(
            Interaction_Split="TEST"
        )
    ],
    ignore_index=True
)


print("\nInteraction membership rows:",
      len(interaction_membership))

print(
    "Duplicate exact keys:",
    int(
        interaction_membership
        .duplicated(KEY)
        .sum()
    )
)

assert len(interaction_membership) == 191785
assert (
    interaction_membership
    .duplicated(KEY)
    .sum()
    ==
    0
)


# canonical metadata can be safely mapped for all unique rows;
# ambiguous rows are NOT used in the primary common-heldout set.
side_meta = (
    side_meta
    .merge(
        interaction_membership,
        on=KEY,
        how="left",
        validate="many_to_one"
    )
)

assert side_meta[
    "Interaction_Split"
].notna().all()


# ============================================================
# 6) SPLIT-CONTAMINATION AUDIT ON EXACT-IDENTITY ROWS
# ============================================================

unique_identity = (
    side_meta["Candidate_Count"]
    ==
    1
)


split_audit_unique = (
    side_meta.loc[
        unique_identity,
        "Interaction_Split"
    ]
    .value_counts()
    .rename_axis(
        "Interaction_Split"
    )
    .reset_index(
        name="Rows"
    )
)

split_audit_unique[
    "Percent_of_Unique_Identity_SideTest"
] = (
    100
    *
    split_audit_unique["Rows"]
    /
    unique_identity.sum()
)


print("\n" + "=" * 120)
print("SIDE-EFFECT TEST → INTERACTION SPLIT AUDIT")
print("(27,345 molecularly exact / non-ambiguous rows only)")
print("=" * 120)

print(
    split_audit_unique.to_string(
        index=False
    )
)


# ============================================================
# 7) PRIMARY COMMON HELD-OUT MASK
# ============================================================

common_mask = (
    unique_identity
    &
    (
        side_meta["Interaction_Split"]
        ==
        "TEST"
    )
)


common_rows = side_meta.loc[
    common_mask
].copy()


COMMON_N = len(
    common_rows
)


print("\n" + "=" * 120)
print("PRIMARY COMMON HELD-OUT SET")
print("=" * 120)

print(
    "Rows held out from BOTH models:",
    COMMON_N
)

print(
    "Percent of full SideEffect test:",
    f"{100*COMMON_N/28768:.4f}%"
)


if COMMON_N < 1000:

    raise RuntimeError(
        "Common held-out set unexpectedly small. "
        "Stop and inspect split mapping."
    )


common_side_indices = (
    common_rows[
        "side_test_row"
    ]
    .astype(int)
    .values
)


# ============================================================
# 8) VERIFY EXACT INTERACTION RECIPE
# ============================================================

with open(
    RECIPE_PATH,
    "r",
    encoding="utf-8"
) as f:

    recipe = json.load(f)


print("\n" + "=" * 120)
print("EXACT INTERACTION RECIPE")
print("=" * 120)

print(
    json.dumps(
        recipe,
        indent=2
    )
)


assert recipe[
    "recipe_exact"
] is True

assert recipe[
    "graph"
] == "TRAIN_BIDIR_UNIQUE"

assert recipe[
    "activation"
] == "relu"

assert recipe[
    "norm_order"
] == "bn_then_act"

assert recipe[
    "final_activation"
] is False

assert (
    abs(
        recipe[
            "cal_macro_f1"
        ]
        -
        0.9086925118104249
    )
    <
    1e-10
)


# ============================================================
# 9) LOAD NODE EMBEDDINGS + FEATURE DRUG ORDER
# ============================================================

H = np.load(
    NODE_EMBEDDINGS_PATH
).astype(
    np.float32
)

cache = np.load(
    FEATURE_CACHE_PATH,
    allow_pickle=True
)

all_drugs = cache[
    "all_drugs"
].astype(str)


print("\nEmbeddings:", H.shape)
print("Drug order:", all_drugs.shape)

assert H.shape == (1705, 192)
assert len(all_drugs) == 1705


drug_to_idx = {
    smi: i
    for i, smi in enumerate(
        all_drugs
    )
}


missing_common_drugs = (
    set(
        common_rows["Drug1"].astype(str)
    )
    |
    set(
        common_rows["Drug2"].astype(str)
    )
) - set(all_drugs)


print(
    "Missing common-heldout drugs:",
    len(
        missing_common_drugs
    )
)

assert len(missing_common_drugs) == 0


# ============================================================
# 10) LOAD INTERACTION CHECKPOINT
# ============================================================

try:

    ckpt = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False
    )

except TypeError:

    ckpt = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu"
    )


assert ckpt["num_classes"] == 84

assert list(
    map(
        str,
        ckpt["label_names"]
    )
) == (
    interaction_mapping[
        "Label"
    ]
    .astype(str)
    .tolist()
)


# ============================================================
# 11) RECONSTRUCT EXACT PAIR CLASSIFIER ONLY
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


classifier = nn.Sequential(

    nn.LayerNorm(
        384
    ),                  # 0

    nn.Linear(
        384,
        512
    ),                  # 1

    nn.ReLU(),          # 2

    nn.Dropout(
        0.0
    ),                  # 3

    nn.Linear(
        512,
        256
    ),                  # 4

    nn.ReLU(),          # 5

    nn.Dropout(
        0.0
    ),                  # 6

    nn.Linear(
        256,
        84
    )                   # 7

).to(
    DEVICE
)


classifier_state = {
    key.replace(
        "classifier.",
        "",
        1
    ): value
    for key, value in ckpt[
        "model_state"
    ].items()
    if key.startswith(
        "classifier."
    )
}


load_result = classifier.load_state_dict(
    classifier_state,
    strict=True
)

print("\nClassifier state:")
print(load_result)

classifier.eval()


# ============================================================
# 12) INTERACTION INFERENCE FUNCTION
# ============================================================

H_tensor = torch.tensor(
    H,
    dtype=torch.float32,
    device=DEVICE
)


@torch.no_grad()
def predict_interaction_pairs(
    df,
    batch_size=8192
):

    u = np.asarray(
        [
            drug_to_idx[
                str(x)
            ]
            for x in df[
                "Drug1"
            ]
        ],
        dtype=np.int64
    )

    v = np.asarray(
        [
            drug_to_idx[
                str(x)
            ]
            for x in df[
                "Drug2"
            ]
        ],
        dtype=np.int64
    )


    predictions = []
    confidences = []


    for start in range(
        0,
        len(df),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(df)
        )


        u_b = torch.tensor(
            u[start:end],
            dtype=torch.long,
            device=DEVICE
        )

        v_b = torch.tensor(
            v[start:end],
            dtype=torch.long,
            device=DEVICE
        )


        pair_repr = torch.cat(
            [
                H_tensor[u_b],
                H_tensor[v_b]
            ],
            dim=1
        )


        logits = classifier(
            pair_repr
        )


        prob = torch.softmax(
            logits,
            dim=1
        )


        conf, pred = torch.max(
            prob,
            dim=1
        )


        predictions.append(
            pred.cpu().numpy()
        )

        confidences.append(
            conf.cpu().numpy()
        )


    return (
        np.concatenate(
            predictions
        ).astype(int),

        np.concatenate(
            confidences
        ).astype(
            np.float32
        )
    )


# ============================================================
# 13) INTERACTION PREDICTIONS ON COMMON HELD-OUT
# ============================================================

int_pred_common, int_conf_common = (
    predict_interaction_pairs(
        common_rows
    )
)


true_int_common = np.asarray(
    [
        label_to_learned_id[
            str(x)
        ]
        for x in common_rows[
            "Label"
        ]
    ],
    dtype=int
)


int_correct_common = (
    int_pred_common
    ==
    true_int_common
)


interaction_common_accuracy = (
    int_correct_common.mean()
)


print("\n" + "=" * 120)
print("INTERACTION — COMMON HELD-OUT")
print("=" * 120)

print(
    f"Accuracy: "
    f"{100*interaction_common_accuracy:.4f}%"
)


# ============================================================
# 14) LOAD PRIMARY SIDE-EFFECT PROBABILITIES
# ============================================================

Y_cal = np.load(
    Y_CAL_PATH
).astype(
    np.uint8
)

Y_test = np.load(
    Y_TEST_PATH
).astype(
    np.uint8
)


P_cal = None
P_test = None


for seed in [
    42,
    43,
    44
]:

    cal_p = np.load(
        BASELINE_ROOT /
        f"seed_{seed}" /
        "calibration_probabilities.npy"
    ).astype(
        np.float32
    )

    test_p = np.load(
        BASELINE_ROOT /
        f"seed_{seed}" /
        "test_probabilities.npy"
    ).astype(
        np.float32
    )


    assert cal_p.shape == (
        28768,
        263
    )

    assert test_p.shape == (
        28768,
        263
    )


    if P_cal is None:

        P_cal = cal_p / 3.0
        P_test = test_p / 3.0

    else:

        P_cal += cal_p / 3.0
        P_test += test_p / 3.0


print(
    "\n Original SideEffect seeds "
    "42 / 43 / 44 averaged equally"
)


# ============================================================
# 15) THRESHOLD SELECTION — CALIBRATION ONLY
# ============================================================

GLOBAL_GRID = np.round(
    np.arange(
        0.05,
        0.851,
        0.01
    ),
    2
)

PER_LABEL_GRID = np.round(
    np.arange(
        0.01,
        0.951,
        0.01
    ),
    2
)


def threshold_counts(
    P,
    Y,
    thresholds
):

    thresholds = np.asarray(
        thresholds,
        dtype=float
    )

    n_samples, n_labels = P.shape
    n_t = len(thresholds)

    TP = np.zeros(
        (
            n_labels,
            n_t
        ),
        dtype=np.int64
    )

    FP = np.zeros_like(TP)
    FN = np.zeros_like(TP)

    support = (
        Y.sum(
            axis=0
        )
        .astype(
            np.int64
        )
    )


    for j in range(n_labels):

        p = P[:, j]

        y = Y[
            :,
            j
        ].astype(
            np.int8
        )

        order = np.argsort(
            -p
        )

        ps = p[order]
        ys = y[order]

        cum_tp = np.cumsum(
            ys,
            dtype=np.int64
        )

        k = np.searchsorted(
            -ps,
            -thresholds,
            side="right"
        )


        tp = np.zeros(
            n_t,
            dtype=np.int64
        )

        mask = (
            k > 0
        )

        tp[mask] = (
            cum_tp[
                k[mask] - 1
            ]
        )

        fp = (
            k.astype(
                np.int64
            )
            -
            tp
        )

        fn = (
            support[j]
            -
            tp
        )

        TP[j] = tp
        FP[j] = fp
        FN[j] = fn


    return TP, FP, FN


def f1_counts(
    tp,
    fp,
    fn
):

    denom = (
        2 * tp
        +
        fp
        +
        fn
    )

    out = np.zeros(
        np.shape(denom),
        dtype=float
    )

    mask = (
        denom > 0
    )

    out[mask] = (
        2
        *
        tp[mask]
        /
        denom[mask]
    )

    return out


# ----- global fallback threshold -----

TPg, FPg, FNg = threshold_counts(
    P_cal,
    Y_cal,
    GLOBAL_GRID
)

label_f1_global = f1_counts(
    TPg,
    FPg,
    FNg
)

macro_global = (
    label_f1_global.mean(
        axis=0
    )
)

micro_global = f1_counts(
    TPg.sum(axis=0),
    FPg.sum(axis=0),
    FNg.sum(axis=0)
)

balanced_global = (
    0.5 * micro_global
    +
    0.5 * macro_global
)

best_global_idx = int(
    np.argmax(
        balanced_global
    )
)

BEST_GLOBAL = float(
    GLOBAL_GRID[
        best_global_idx
    ]
)


print(
    "\nBest calibration-only global threshold:",
    BEST_GLOBAL
)

assert abs(
    BEST_GLOBAL
    -
    0.54
) < 1e-12


# ----- per-label -----

TPl, FPl, FNl = threshold_counts(
    P_cal,
    Y_cal,
    PER_LABEL_GRID
)

label_f1_grid = f1_counts(
    TPl,
    FPl,
    FNl
)

cal_support = (
    Y_cal.sum(
        axis=0
    )
    .astype(int)
)

thresholds = np.full(
    263,
    BEST_GLOBAL,
    dtype=float
)


for j in range(263):

    if cal_support[j] < 3:
        continue

    idx = int(
        np.argmax(
            label_f1_grid[j]
        )
    )

    thresholds[j] = float(
        PER_LABEL_GRID[idx]
    )


side_pred = (
    P_test
    >=
    thresholds[
        None,
        :
    ]
).astype(
    np.uint8
)


# ============================================================
# 16) VERIFY PRIMARY SIDE-EFFECT SIGNATURE
# ============================================================

def side_metrics(
    Y,
    pred
):

    return {

        "Exact":
            accuracy_score(
                Y,
                pred
            ),

        "Micro_P":
            precision_score(
                Y,
                pred,
                average="micro",
                zero_division=0
            ),

        "Micro_R":
            recall_score(
                Y,
                pred,
                average="micro",
                zero_division=0
            ),

        "Micro_F1":
            f1_score(
                Y,
                pred,
                average="micro",
                zero_division=0
            ),

        "Macro_P":
            precision_score(
                Y,
                pred,
                average="macro",
                zero_division=0
            ),

        "Macro_R":
            recall_score(
                Y,
                pred,
                average="macro",
                zero_division=0
            ),

        "Macro_F1":
            f1_score(
                Y,
                pred,
                average="macro",
                zero_division=0
            ),

        "Weighted_F1":
            f1_score(
                Y,
                pred,
                average="weighted",
                zero_division=0
            ),

        "Hamming":
            hamming_loss(
                Y,
                pred
            )
    }


side_full_metrics = side_metrics(
    Y_test,
    side_pred
)


print("\n" + "=" * 120)
print("PRIMARY SIDE-EFFECT SIGNATURE")
print("=" * 120)

for k, v in side_full_metrics.items():

    if k == "Hamming":
        print(
            f"{k:15s}: "
            f"{v:.8f}"
        )

    else:
        print(
            f"{k:15s}: "
            f"{100*v:.4f}%"
        )


# Known final primary signature
assert abs(
    side_full_metrics[
        "Exact"
    ]
    -
    0.7642171857619577
) < 2e-6

assert abs(
    side_full_metrics[
        "Micro_F1"
    ]
    -
    0.916328
) < 3e-6

assert abs(
    side_full_metrics[
        "Macro_F1"
    ]
    -
    0.920570
) < 3e-6


print(
    "\n Final primary SideEffect model confirmed"
)


# ============================================================
# 17) SIDE-EFFECT EXACT CORRECTNESS
# ============================================================

side_correct_all = np.all(
    side_pred
    ==
    Y_test,
    axis=1
)


print(
    "Full SideEffect exact-correct rows:",
    int(
        side_correct_all.sum()
    ),
    "/ 28768"
)


assert int(
    side_correct_all.sum()
) == 21985


side_correct_common = (
    side_correct_all[
        common_side_indices
    ]
)


side_common_exact = (
    side_correct_common.mean()
)


# ============================================================
# 18) FOUR INTEGRATED CATEGORIES
# ============================================================

both_correct = (
    int_correct_common
    &
    side_correct_common
)

interaction_only = (
    int_correct_common
    &
    (~side_correct_common)
)

side_only = (
    (~int_correct_common)
    &
    side_correct_common
)

both_wrong = (
    (~int_correct_common)
    &
    (~side_correct_common)
)


category_counts = {

    "Both correct":
        int(
            both_correct.sum()
        ),

    "Interaction only correct":
        int(
            interaction_only.sum()
        ),

    "Side-effects only correct":
        int(
            side_only.sum()
        ),

    "Both wrong":
        int(
            both_wrong.sum()
        )
}


assert sum(
    category_counts.values()
) == COMMON_N


four_categories = pd.DataFrame(
    {
        "Category":
            list(
                category_counts.keys()
            ),

        "Count":
            list(
                category_counts.values()
            )
    }
)


four_categories[
    "Percent"
] = (
    100
    *
    four_categories["Count"]
    /
    COMMON_N
)


STRICT_INTEGRATED_ACCURACY = (
    both_correct.mean()
)


print("\n" + "=" * 120)
print("PRIMARY INTEGRATED RESULTS")
print("COMMON HELD-OUT FROM BOTH MODELS")
print("=" * 120)

print(
    "N:",
    COMMON_N
)

print(
    f"Interaction accuracy : "
    f"{100*interaction_common_accuracy:.4f}%"
)

print(
    f"SideEffect exact     : "
    f"{100*side_common_exact:.4f}%"
)

print(
    f"STRICT integrated    : "
    f"{100*STRICT_INTEGRATED_ACCURACY:.4f}%"
)

print("\nFour categories:")

print(
    four_categories.to_string(
        index=False
    )
)


# ============================================================
# 19) BUILD TRAIN-ONLY INTERACTION → SIDEEFFECT LOOKUP
# ============================================================

side_label_map = (
    pd.read_csv(
        SIDE_LABEL_MAP_PATH
    )
    .sort_values("label_id")
    .reset_index(drop=True)
)


assert len(side_label_map) == 263


side_label_to_id = dict(
    zip(
        side_label_map[
            "label"
        ].astype(str),

        side_label_map[
            "label_id"
        ].astype(int)
    )
)


def effect_string_to_ids(
    value
):

    terms = [
        x.strip()
        for x in str(
            value
        ).split("|")
        if x.strip()
    ]

    unknown = [
        x
        for x in terms
        if x not in side_label_to_id
    ]

    if unknown:
        raise KeyError(
            f"Unknown side-effect terms "
            f"in train lookup: {unknown}"
        )

    return tuple(
        sorted(
            side_label_to_id[x]
            for x in terms
        )
    )


train_rows = train_rows.copy()

train_rows[
    "_target_ids"
] = (
    train_rows[
        "sideeffect_str"
    ]
    .map(
        effect_string_to_ids
    )
)


lookup_audit = (
    train_rows
    .groupby(
        "Label"
    )
    .agg(
        Train_Rows=(
            "_target_ids",
            "size"
        ),

        Unique_Target_Vectors=(
            "_target_ids",
            "nunique"
        )
    )
    .reset_index()
)


print("\n" + "=" * 120)
print("TRAIN-ONLY LOOKUP AUDIT")
print("=" * 120)

print(
    "Interaction labels in lookup:",
    lookup_audit[
        "Label"
    ].nunique()
)

print(
    "Classes with one target vector:",
    int(
        (
            lookup_audit[
                "Unique_Target_Vectors"
            ]
            ==
            1
        ).sum()
    )
)

print(
    "Classes with >1 target vectors:",
    int(
        (
            lookup_audit[
                "Unique_Target_Vectors"
            ]
            >
            1
        ).sum()
    )
)


assert lookup_audit[
    "Label"
].nunique() == 84

assert (
    lookup_audit[
        "Unique_Target_Vectors"
    ]
    ==
    1
).all()


# ============================================================
# 20) LOOKUP MATRIX: LEARNED INTERACTION CLASS → 263 TARGETS
# ============================================================

lookup_matrix = np.zeros(
    (
        84,
        263
    ),
    dtype=np.uint8
)


for learned_id in range(
    84
):

    exact_label = learned_id_to_label[
        learned_id
    ]


    rows = train_rows[
        train_rows[
            "Label"
        ].astype(str)
        ==
        exact_label
    ]


    assert len(rows) > 0


    unique_targets = rows[
        "_target_ids"
    ].unique()


    assert len(
        unique_targets
    ) == 1


    ids = unique_targets[
        0
    ]


    lookup_matrix[
        learned_id,
        list(ids)
    ] = 1


# ============================================================
# 21) ORACLE TRUE-INTERACTION LOOKUP AUDIT
# ============================================================

oracle_lookup_pred = (
    lookup_matrix[
        true_int_common
    ]
)


oracle_lookup_correct = np.all(
    oracle_lookup_pred
    ==
    Y_test[
        common_side_indices
    ],
    axis=1
)


print("\n" + "=" * 120)
print("STRUCTURAL DEPENDENCE AUDIT")
print("=" * 120)

print(
    "True Interaction → train lookup "
    "SideEffect exact:"
)

print(
    int(
        oracle_lookup_correct.sum()
    ),
    "/",
    COMMON_N,
    "=",
    f"{100*oracle_lookup_correct.mean():.4f}%"
)


# This should be 100% if each interaction class
# deterministically defines its side-effect target vector.
assert oracle_lookup_correct.all()


# ============================================================
# 22) PREDICTED-INTERACTION TRAIN LOOKUP BASELINE
# ============================================================

lookup_side_pred = (
    lookup_matrix[
        int_pred_common
    ]
)


lookup_side_correct = np.all(
    lookup_side_pred
    ==
    Y_test[
        common_side_indices
    ],
    axis=1
)


lookup_side_exact = (
    lookup_side_correct.mean()
)


lookup_both_correct = (
    int_correct_common
    &
    lookup_side_correct
)


lookup_strict = (
    lookup_both_correct.mean()
)


print("\n" + "=" * 120)
print("TRAIN-LOOKUP BASELINE")
print("=" * 120)

print(
    "Input:",
    "Predicted Interaction class"
)

print(
    "Output:",
    "SideEffect vector learned from Interaction training rows"
)

print(
    f"Lookup SideEffect exact : "
    f"{100*lookup_side_exact:.4f}%"
)

print(
    f"Lookup strict integrated: "
    f"{100*lookup_strict:.4f}%"
)


# ============================================================
# 23) LOOKUP FOUR-CATEGORY TABLE
# ============================================================

lookup_categories = pd.DataFrame(
    {
        "Category": [
            "Both correct",
            "Interaction only correct",
            "Side-effects only correct",
            "Both wrong"
        ],

        "Count": [
            int(
                (
                    int_correct_common
                    &
                    lookup_side_correct
                ).sum()
            ),

            int(
                (
                    int_correct_common
                    &
                    (~lookup_side_correct)
                ).sum()
            ),

            int(
                (
                    (~int_correct_common)
                    &
                    lookup_side_correct
                ).sum()
            ),

            int(
                (
                    (~int_correct_common)
                    &
                    (~lookup_side_correct)
                ).sum()
            )
        ]
    }
)


lookup_categories[
    "Percent"
] = (
    100
    *
    lookup_categories["Count"]
    /
    COMMON_N
)


# ============================================================
# 24) DIRECT MODEL vs LOOKUP BASELINE
# ============================================================

comparison = pd.DataFrame(
    {
        "Method": [
            "Independent SideEffect LGBM ensemble",
            "Predicted-Interaction Train Lookup"
        ],

        "Interaction_Accuracy": [
            interaction_common_accuracy,
            interaction_common_accuracy
        ],

        "SideEffect_Exact_Match": [
            side_common_exact,
            lookup_side_exact
        ],

        "Strict_Integrated_Accuracy": [
            STRICT_INTEGRATED_ACCURACY,
            lookup_strict
        ],

        "Evaluation_Rows": [
            COMMON_N,
            COMMON_N
        ]
    }
)


comparison[
    "SideEffect_Exact_Percent"
] = (
    100
    *
    comparison[
        "SideEffect_Exact_Match"
    ]
)


comparison[
    "Strict_Integrated_Percent"
] = (
    100
    *
    comparison[
        "Strict_Integrated_Accuracy"
    ]
)


print("\n" + "=" * 120)
print("FINAL MODEL vs TRAIN LOOKUP")
print("=" * 120)

print(
    comparison.to_string(
        index=False
    )
)


# ============================================================
# 25) ROW-LEVEL OUTPUT
# ============================================================

row_output = common_rows[
    [
        "side_test_row",
        "full_dataset_row",
        "Drug1_ID",
        "Drug1",
        "Drug2_ID",
        "Drug2",
        "Interaction",
        "Label"
    ]
].copy()


row_output[
    "True_Learned_Interaction_ID"
] = true_int_common


row_output[
    "Pred_Learned_Interaction_ID"
] = int_pred_common


row_output[
    "Pred_Interaction_Label"
] = [
    learned_id_to_label[
        int(x)
    ]
    for x in int_pred_common
]


row_output[
    "Interaction_Confidence"
] = int_conf_common


row_output[
    "Interaction_Correct"
] = int_correct_common


row_output[
    "SideEffect_Exact_Correct"
] = side_correct_common


row_output[
    "Integrated_Category"
] = np.select(
    [
        both_correct,
        interaction_only,
        side_only,
        both_wrong
    ],
    [
        "Both correct",
        "Interaction only correct",
        "Side-effects only correct",
        "Both wrong"
    ],
    default="ERROR"
)


row_output[
    "Lookup_SideEffect_Exact_Correct"
] = lookup_side_correct


row_output[
    "Lookup_Strict_Correct"
] = lookup_both_correct


# ============================================================
# 26) INTERACTION CLASS-WISE COMMON TEST PERFORMANCE
# ============================================================

class_rows = []


for class_id in sorted(
    np.unique(
        true_int_common
    )
):

    mask = (
        true_int_common
        ==
        class_id
    )


    class_rows.append(
        {
            "label_id":
                int(
                    class_id
                ),

            "Interaction_Label":
                learned_id_to_label[
                    int(
                        class_id
                    )
                ],

            "Support":
                int(
                    mask.sum()
                ),

            "Interaction_Accuracy":
                float(
                    int_correct_common[
                        mask
                    ].mean()
                ),

            "SideEffect_Exact":
                float(
                    side_correct_common[
                        mask
                    ].mean()
                ),

            "Strict_Integrated":
                float(
                    (
                        int_correct_common[
                            mask
                        ]
                        &
                        side_correct_common[
                            mask
                        ]
                    ).mean()
                ),

            "Lookup_SideEffect_Exact":
                float(
                    lookup_side_correct[
                        mask
                    ].mean()
                )
        }
    )


classwise = pd.DataFrame(
    class_rows
)


# ============================================================
# 27) QA TABLE
# ============================================================

qa = pd.DataFrame(
    {
        "QA_Check": [
            "Interaction recipe exactly recovered",
            "Interaction recipe uses TRAIN_BIDIR_UNIQUE",
            "SideEffect primary ensemble signature confirmed",
            "Full SideEffect exact-correct count = 21985",
            "Only exact-identity SideEffect rows used in primary integrated set",
            "Every primary row belongs to Interaction TEST",
            "No Interaction TRAIN rows in primary integrated set",
            "No Interaction CAL rows in primary integrated set",
            "Train lookup covers all 84 Interaction classes",
            "Each train lookup class has one target vector",
            "True-Interaction oracle lookup = 100%",
            "Four primary categories sum to evaluation N",
            "No training performed"
        ],

        "Passed": [
            recipe[
                "recipe_exact"
            ] is True,

            recipe[
                "graph"
            ]
            ==
            "TRAIN_BIDIR_UNIQUE",

            abs(
                side_full_metrics[
                    "Exact"
                ]
                -
                0.7642171857619577
            )
            <
            2e-6,

            int(
                side_correct_all.sum()
            )
            ==
            21985,

            bool(
                (
                    common_rows[
                        "Candidate_Count"
                    ]
                    ==
                    1
                ).all()
            ),

            bool(
                (
                    common_rows[
                        "Interaction_Split"
                    ]
                    ==
                    "TEST"
                ).all()
            ),

            bool(
                (
                    common_rows[
                        "Interaction_Split"
                    ]
                    ==
                    "TRAIN"
                ).sum()
                ==
                0
            ),

            bool(
                (
                    common_rows[
                        "Interaction_Split"
                    ]
                    ==
                    "CAL"
                ).sum()
                ==
                0
            ),

            lookup_audit[
                "Label"
            ].nunique()
            ==
            84,

            bool(
                (
                    lookup_audit[
                        "Unique_Target_Vectors"
                    ]
                    ==
                    1
                ).all()
            ),

            bool(
                oracle_lookup_correct.all()
            ),

            sum(
                category_counts.values()
            )
            ==
            COMMON_N,

            True
        ]
    }
)


print("\n" + "=" * 120)
print("FINAL QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)

assert qa[
    "Passed"
].all()


# ============================================================
# 28) SUMMARY TABLE
# ============================================================

summary_table = pd.DataFrame(
    {
        "Metric": [
            "Primary common-heldout rows",
            "Interaction accuracy",
            "Independent SideEffect exact match",
            "Strict integrated accuracy",
            "Both correct",
            "Interaction only correct",
            "Side-effects only correct",
            "Both wrong",
            "Train-lookup SideEffect exact",
            "Train-lookup strict integrated",
            "True-Interaction oracle lookup exact"
        ],

        "Value": [
            COMMON_N,
            interaction_common_accuracy,
            side_common_exact,
            STRICT_INTEGRATED_ACCURACY,
            category_counts[
                "Both correct"
            ],
            category_counts[
                "Interaction only correct"
            ],
            category_counts[
                "Side-effects only correct"
            ],
            category_counts[
                "Both wrong"
            ],
            lookup_side_exact,
            lookup_strict,
            oracle_lookup_correct.mean()
        ]
    }
)


# ============================================================
# 29) SAVE CSVs
# ============================================================

summary_table.to_csv(
    OUT /
    "FINAL_Integrated_Summary.csv",
    index=False
)

four_categories.to_csv(
    OUT /
    "FINAL_Four_Categories.csv",
    index=False
)

lookup_categories.to_csv(
    OUT /
    "FINAL_Lookup_Four_Categories.csv",
    index=False
)

comparison.to_csv(
    OUT /
    "FINAL_Model_vs_TrainLookup.csv",
    index=False
)

row_output.to_csv(
    OUT /
    "FINAL_Common_Heldout_Row_Level.csv",
    index=False
)

classwise.to_csv(
    OUT /
    "FINAL_Common_Heldout_Classwise.csv",
    index=False
)

split_audit_unique.to_csv(
    OUT /
    "FINAL_Split_Contamination_Audit.csv",
    index=False
)

lookup_audit.to_csv(
    OUT /
    "FINAL_Train_Lookup_Audit.csv",
    index=False
)

qa.to_csv(
    OUT /
    "FINAL_QA.csv",
    index=False
)


# ============================================================
# 30) EXCEL
# ============================================================

XLSX_PATH = (
    OUT /
    "FINAL_INTEGRATED_COMMON_HELDOUT_RESULTS.xlsx"
)


with pd.ExcelWriter(
    XLSX_PATH,
    engine="openpyxl"
) as writer:

    summary_table.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    comparison.to_excel(
        writer,
        sheet_name="Model_vs_Lookup",
        index=False
    )

    four_categories.to_excel(
        writer,
        sheet_name="Four_Categories",
        index=False
    )

    lookup_categories.to_excel(
        writer,
        sheet_name="Lookup_Categories",
        index=False
    )

    split_audit_unique.to_excel(
        writer,
        sheet_name="Split_Audit",
        index=False
    )

    classwise.to_excel(
        writer,
        sheet_name="Classwise",
        index=False
    )

    lookup_audit.to_excel(
        writer,
        sheet_name="Lookup_Audit",
        index=False
    )

    row_output.to_excel(
        writer,
        sheet_name="Row_Level",
        index=False
    )

    qa.to_excel(
        writer,
        sheet_name="QA",
        index=False
    )


# ============================================================
# 31) SUMMARY JSON
# ============================================================

summary_json = {

    "evaluation_protocol":
        "Exact-identity intersection of SideEffect TEST and Interaction TEST",

    "common_heldout_rows":
        int(
            COMMON_N
        ),

    "full_sideeffect_test_rows":
        28768,

    "interaction_accuracy":
        float(
            interaction_common_accuracy
        ),

    "independent_sideeffect_exact_match":
        float(
            side_common_exact
        ),

    "strict_integrated_accuracy":
        float(
            STRICT_INTEGRATED_ACCURACY
        ),

    "four_categories":
        category_counts,

    "train_lookup_sideeffect_exact_match":
        float(
            lookup_side_exact
        ),

    "train_lookup_strict_integrated_accuracy":
        float(
            lookup_strict
        ),

    "true_interaction_oracle_lookup_exact":
        float(
            oracle_lookup_correct.mean()
        ),

    "excluded_morgan_ambiguous_side_test_rows":
        1423,

    "test_used_for_threshold_selection":
        False,

    "training_performed":
        False
}


with open(
    OUT /
    "FINAL_INTEGRATED_SUMMARY.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary_json,
        f,
        indent=2
    )


# ============================================================
# 32) ZIP
# ============================================================

ZIP_PATH = shutil.make_archive(
    "/kaggle/working/"
    "FINAL_INTEGRATED_COMMON_HELDOUT_EVALUATION",
    "zip",
    root_dir=OUT
)


# ============================================================
# 33) FINAL PRINT
# ============================================================

print("\n" + "=" * 120)
print("FINAL INTEGRATED RESULTS")
print("=" * 120)

print(
    f"Common held-out N       : "
    f"{COMMON_N:,}"
)

print(
    f"Interaction accuracy    : "
    f"{100*interaction_common_accuracy:.4f}%"
)

print(
    f"SideEffect Exact Match  : "
    f"{100*side_common_exact:.4f}%"
)

print(
    f"STRICT Integrated Acc.  : "
    f"{100*STRICT_INTEGRATED_ACCURACY:.4f}%"
)


print("\nFOUR CATEGORIES")

print(
    four_categories.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("MODEL vs TRAIN LOOKUP")
print("=" * 120)

print(
    comparison.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("STRUCTURAL DEPENDENCE")
print("=" * 120)

print(
    "True Interaction → Train Lookup:",
    f"{100*oracle_lookup_correct.mean():.4f}%"
)


print("\nExcel:")
print(
    XLSX_PATH
)

print("\nZIP:")
print(
    ZIP_PATH
)

print("\n Interaction recipe exactly reproduced")
print(" Original SideEffect ensemble used")
print(" Primary evaluation held out from BOTH models")
print(" Morgan-ambiguous rows excluded from headline metric")
print(" Same rows used for model-vs-lookup comparison")
print(" NO TRAINING PERFORMED")